# DocMind — Retrieval Research Benchmark (Clean Corpus)

This notebook is the new research track for **text-document retrieval only**.

## Goal
Measure which combination of:
- extraction baseline,
- chunk size,
- embedding model,
- BM25 / dense / hybrid retrieval,
- reranking,

retrieves the **correct evidence passage** most reliably.

## Deliberately excluded
- LLM answer generation
- LLM-as-a-judge
- SQL routing
- giant CSV datasets
- table-heavy grade spreadsheets
- OCR benchmarking

Those become separate notebooks later.

## Experimental rule
**Do not choose a winner using the same questions used for final reporting.**
We will use a development split for configuration selection and a held-out test split for the final result.

In [12]:
from pathlib import Path
import json

# Find frozen DEV file
DEV_PATH = list(
    Path("/kaggle/input").rglob(
        "docmind_generation_dev_frozen.json"
    )
)[0]

with open(
    DEV_PATH,
    "r",
    encoding="utf-8"
) as f:
    DEV_DATA = json.load(f)

DEV_IDS = {
    r["question_id"]
    for r in DEV_DATA["records"]
}

print("DEV IDs:", len(DEV_IDS))
assert len(DEV_IDS) == 46

DEV IDs: 46


In [16]:
from pathlib import Path
import json

KAGGLE_INPUT = Path("/kaggle/input")

# Try the filenames we've used before
candidates = []

for name in [
    "rag_gold_questions.json",
    "rag_gold_questions_starter_v1.json",
]:
    candidates.extend(
        KAGGLE_INPUT.rglob(name)
    )

if not candidates:
    raise FileNotFoundError(
        "Could not find the gold questions JSON in /kaggle/input.\n"
        "Attach the Kaggle dataset containing rag_gold_questions.json."
    )

GOLD_PATH = candidates[0]

print("Using gold file:")
print(GOLD_PATH)

with open(
    GOLD_PATH,
    "r",
    encoding="utf-8"
) as f:
    gold_raw = json.load(f)

# Handle either:
# [ {...}, {...} ]
# or
# {"questions": [...]}
# or
# {"records": [...]}
if isinstance(gold_raw, list):
    gold_questions = gold_raw

elif isinstance(gold_raw, dict):
    gold_questions = (
        gold_raw.get("questions")
        or gold_raw.get("records")
    )

else:
    raise TypeError(
        "Unknown gold JSON format"
    )

assert isinstance(gold_questions, list)

print("\n✅ gold_questions restored")
print("Total questions:", len(gold_questions))

assert len(gold_questions) == 62

Using gold file:
/kaggle/input/datasets/ripperdzz/past-session/rag_gold_questions.json

✅ gold_questions restored
Total questions: 62


In [17]:
answerable_questions = [
    q for q in gold_questions
    if q.get("answerable", False)
]

test_questions = [
    q for q in answerable_questions
    if q["question_id"] not in DEV_IDS
]

print("Recovered TEST:", len(test_questions))

assert len(test_questions) == 12

Recovered TEST: 12


In [18]:
from pathlib import Path
import json

DEV_PATH = list(
    Path("/kaggle/input").rglob(
        "docmind_generation_dev_frozen.json"
    )
)[0]

with open(
    DEV_PATH,
    "r",
    encoding="utf-8"
) as f:
    DEV_DATA = json.load(f)

DEV_IDS = {
    r["question_id"]
    for r in DEV_DATA["records"]
}

answerable_questions = [
    q for q in gold_questions
    if q.get("answerable", False)
]

test_questions = [
    q for q in answerable_questions
    if q["question_id"] not in DEV_IDS
]

no_answer_questions = [
    q for q in gold_questions
    if not q.get("answerable", False)
]

print("DEV IDs   :", len(DEV_IDS))
print("TEST      :", len(test_questions))
print("NO-ANSWER :", len(no_answer_questions))

assert len(DEV_IDS) == 46
assert len(test_questions) == 12
assert len(no_answer_questions) == 4

DEV IDs   : 46
TEST      : 12
NO-ANSWER : 4


In [19]:
from pathlib import Path
import json

OUTPUT_DIR = Path("/kaggle/working/docmind_final_v1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_QUESTIONS_PATH = OUTPUT_DIR / "docmind_test_questions.json"

with open(TEST_QUESTIONS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        test_questions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✅ Saved:")
print(TEST_QUESTIONS_PATH)
print("Questions:", len(test_questions))

assert TEST_QUESTIONS_PATH.exists()
assert len(test_questions) == 12

✅ Saved:
/kaggle/working/docmind_final_v1/docmind_test_questions.json
Questions: 12


In [58]:
# ============================================================
# KAGGLE DATASET INPUT LOADER — DOCMIND
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd


KAGGLE_INPUT = Path("/kaggle/input")


def find_input_file(filename, required=True):
    """
    Find a file anywhere under /kaggle/input.

    Returns the first exact filename match.
    """

    matches = list(
        KAGGLE_INPUT.rglob(filename)
    )

    if not matches:
        if required:
            raise FileNotFoundError(
                f"\nCould not find:\n{filename}\n"
                "inside /kaggle/input.\n\n"
                "Make sure the Kaggle dataset containing it "
                "is attached to this notebook."
            )

        return None

    if len(matches) > 1:
        print(
            f"⚠️ Multiple matches for {filename}:"
        )

        for p in matches:
            print("  -", p)

        print(
            "\nUsing first match:"
        )

    path = matches[0]

    print(
        f"✅ {filename}\n   → {path}"
    )

    return path

In [59]:
# ============================================================
# SHOW ALL ATTACHED DOCMIND FILES
# ============================================================

print("=" * 90)
print("FILES AVAILABLE UNDER /kaggle/input")
print("=" * 90)

for path in sorted(
    KAGGLE_INPUT.rglob("*")
):
    if path.is_file():
        print(path)

FILES AVAILABLE UNDER /kaggle/input
/kaggle/input/datasets/ripperdzz/docmind-test/1. Introduction.pdf
/kaggle/input/datasets/ripperdzz/docmind-test/10. HMM.pdf
/kaggle/input/datasets/ripperdzz/docmind-test/11. DTW.pdf
/kaggle/input/datasets/ripperdzz/docmind-test/2. Concepts fondamentaux.pdf
/kaggle/input/datasets/ripperdzz/docmind-test/3. Regression.pdf
/kaggle/input/datasets/ripperdzz/docmind-test/4. Regression Logistique.pdf
/kaggle/input/datasets/ripperdzz/docmind-test/5. Non-paramtriques-KNN.pdf
/kaggle/input/datasets/ripperdzz/docmind-test/6. Naive Bayes.pdf
/kaggle/input/datasets/ripperdzz/docmind-test/7. Clustering - part2.pdf
/kaggle/input/datasets/ripperdzz/docmind-test/7. Clustering-part1.pdf
/kaggle/input/datasets/ripperdzz/docmind-test/8. ANN-part1.pdf
/kaggle/input/datasets/ripperdzz/docmind-test/8. ANN-part2.pdf
/kaggle/input/datasets/ripperdzz/docmind-test/9. SVM.pdf
/kaggle/input/datasets/ripperdzz/docmind-test/AI and Biases.pptx
/kaggle/input/datasets/ripperdzz/docmin

In [60]:
# ============================================================
# LOAD FROZEN RESEARCH INPUTS — NOTEBOOK 02
# ============================================================

GOLD_PATH = find_input_file(
    "rag_gold_questions.json"
)

MEDIUM_CHUNKS_PATH = find_input_file(
    "docmind_chunks_medium.json"
)


with open(
    GOLD_PATH,
    "r",
    encoding="utf-8"
) as f:
    gold_data = json.load(f)


with open(
    MEDIUM_CHUNKS_PATH,
    "r",
    encoding="utf-8"
) as f:
    medium_chunks = json.load(f)


# Some JSONs may contain {"questions": [...]}
# rather than directly being a list.
if isinstance(gold_data, dict):
    gold_questions = gold_data.get(
        "questions",
        gold_data.get("records")
    )
else:
    gold_questions = gold_data


assert isinstance(
    gold_questions,
    list
)

assert isinstance(
    medium_chunks,
    list
)


print("\nGold questions :", len(gold_questions))
print("Medium chunks  :", len(medium_chunks))

✅ rag_gold_questions.json
   → /kaggle/input/datasets/ripperdzz/past-session/rag_gold_questions.json
✅ docmind_chunks_medium.json
   → /kaggle/input/datasets/ripperdzz/past-session/docmind_chunks_medium.json

Gold questions : 62
Medium chunks  : 1210


In [61]:
# ============================================================
# RECOVER EXACT ORIGINAL DEV/TEST SPLIT
# ============================================================

DEV_FROZEN_PATH = find_input_file(
    "docmind_generation_dev_frozen.json"
)


with open(
    DEV_FROZEN_PATH,
    "r",
    encoding="utf-8"
) as f:
    frozen_dev = json.load(f)


frozen_dev_records = frozen_dev[
    "records"
]


DEV_IDS = {
    r["question_id"]
    for r in frozen_dev_records
}


answerable_questions = [
    q for q in gold_questions
    if q.get("answerable", False)
]


dev_questions = [
    q for q in answerable_questions
    if q["question_id"] in DEV_IDS
]


test_questions = [
    q for q in answerable_questions
    if q["question_id"] not in DEV_IDS
]


no_answer_questions = [
    q for q in gold_questions
    if not q.get("answerable", False)
]


print("=" * 70)
print("RECOVERED ORIGINAL SPLIT")
print("=" * 70)

print("DEV       :", len(dev_questions))
print("TEST      :", len(test_questions))
print("NO-ANSWER :", len(no_answer_questions))


assert len(dev_questions) == 46
assert len(test_questions) == 12
assert len(no_answer_questions) == 4

✅ docmind_generation_dev_frozen.json
   → /kaggle/input/datasets/ripperdzz/past-session/docmind_generation_dev_frozen.json
RECOVERED ORIGINAL SPLIT
DEV       : 46
TEST      : 12
NO-ANSWER : 4


In [62]:
# ============================================================
# FIND QWEN MEDIUM DOCUMENT EMBEDDINGS
# ============================================================

def find_qwen_medium_embeddings(
    expected_chunks=1210,
    expected_dim=1024
):

    candidates = []

    for path in KAGGLE_INPUT.rglob("*.npy"):

        try:
            arr = np.load(
                path,
                mmap_mode="r"
            )

            if (
                arr.ndim == 2
                and arr.shape[0] == expected_chunks
                and arr.shape[1] == expected_dim
            ):
                candidates.append(
                    path
                )

        except Exception:
            pass

    if not candidates:
        raise FileNotFoundError(
            "Could not find a 1210 × 1024 "
            "document embedding matrix."
        )

    # Prefer filename containing qwen
    qwen_candidates = [
        p for p in candidates
        if "qwen" in str(p).lower()
    ]

    if qwen_candidates:
        candidates = qwen_candidates

    print(
        "Embedding candidates:"
    )

    for p in candidates:
        print("  -", p)

    selected = candidates[0]

    print(
        "\n✅ Using:",
        selected
    )

    return selected


QWEN_MEDIUM_EMBED_PATH = (
    find_qwen_medium_embeddings(
        expected_chunks=len(
            medium_chunks
        )
    )
)


DOCUMENT_EMBEDDINGS = np.asarray(
    np.load(
        QWEN_MEDIUM_EMBED_PATH
    ),
    dtype=np.float32
)


print(
    "Embedding matrix:",
    DOCUMENT_EMBEDDINGS.shape
)

Embedding candidates:
  - /kaggle/input/datasets/ripperdzz/past-session/qwen3_06b__medium__documents.npy

✅ Using: /kaggle/input/datasets/ripperdzz/past-session/qwen3_06b__medium__documents.npy
Embedding matrix: (1210, 1024)


In [63]:
FINAL_CHUNKS = chunks_by_config["medium"]

In [64]:
# ============================================================
# FINAL FROZEN CHUNKS
# ============================================================

FINAL_CHUNKS = medium_chunks

print(
    "Final chunks:",
    len(FINAL_CHUNKS)
)

assert len(FINAL_CHUNKS) == 1210

Final chunks: 1210


In [65]:
WORK_DIR = Path(
    "/kaggle/working/docmind_final_v1"
)

WORK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(WORK_DIR)

/kaggle/working/docmind_final_v1


In [66]:
# ============================================================
# FINAL RESEARCH V1 — LOCKED CONFIGURATION
# ============================================================

import re
import json
import gc
import numpy as np
import pandas as pd
import faiss

from rank_bm25 import BM25Okapi


FINAL_CHUNK_CONFIG = "medium"

FINAL_EMBEDDING_KEY = (
    "qwen3_06b"
)

FINAL_RERANK_TOP_N = 30
FINAL_TOP_K = 5
FINAL_RRF_K = 60


# Already loaded from Kaggle dataset
FINAL_CHUNKS = medium_chunks


# Use the already recovered split
NOANSWER_QUESTIONS = (
    no_answer_questions
)


assert len(FINAL_CHUNKS) == 1210
assert len(dev_questions) == 46
assert len(test_questions) == 12
assert len(NOANSWER_QUESTIONS) == 4


print("=" * 80)
print("DOCMIND RESEARCH V1 — FINAL FROZEN CONFIG")
print("=" * 80)

print(
    "Embedding      :",
    "Qwen/Qwen3-Embedding-0.6B"
)

print(
    "Chunks         :",
    "320 tokens / 48 overlap"
)

print(
    "Chunk count    :",
    len(FINAL_CHUNKS)
)

print(
    "Retrieval      :",
    "Dense + BM25"
)

print(
    "Fusion         :",
    f"RRF k={FINAL_RRF_K}"
)

print(
    "Reranker       :",
    "BAAI/bge-reranker-v2-m3"
)

print(
    "Rerank top N   :",
    FINAL_RERANK_TOP_N
)

print(
    "Final context K:",
    FINAL_TOP_K
)

print()

print(
    "DEV       :",
    len(dev_questions)
)

print(
    "TEST      :",
    len(test_questions)
)

print(
    "NO-ANSWER :",
    len(NOANSWER_QUESTIONS)
)

print(
    "\n🔒 CONFIGURATION FROZEN"
)

DOCMIND RESEARCH V1 — FINAL FROZEN CONFIG
Embedding      : Qwen/Qwen3-Embedding-0.6B
Chunks         : 320 tokens / 48 overlap
Chunk count    : 1210
Retrieval      : Dense + BM25
Fusion         : RRF k=60
Reranker       : BAAI/bge-reranker-v2-m3
Rerank top N   : 30
Final context K: 5

DEV       : 46
TEST      : 12
NO-ANSWER : 4

🔒 CONFIGURATION FROZEN


In [67]:
# ============================================================
# FINAL TEST + NO-ANSWER RETRIEVAL
# ============================================================

print(
    "Document embeddings:",
    DOCUMENT_EMBEDDINGS.shape
)

assert DOCUMENT_EMBEDDINGS.shape == (
    len(FINAL_CHUNKS),
    1024
)

print(
    "✅ Saved Qwen-medium embeddings loaded correctly"
)

Document embeddings: (1210, 1024)
✅ Saved Qwen-medium embeddings loaded correctly


In [68]:
# ============================================================
# QUERY EMBEDDINGS
# ============================================================

embedding_model = load_embedding_model(
    FINAL_EMBEDDING_KEY
)


TEST_QUERY_EMBEDDINGS = encode_queries(
    embedding_model,
    FINAL_EMBEDDING_KEY,
    [
        q["question"]
        for q in test_questions
    ],
    batch_size=64
)


NOANSWER_QUERY_EMBEDDINGS = encode_queries(
    embedding_model,
    FINAL_EMBEDDING_KEY,
    [
        q["question"]
        for q in NOANSWER_QUESTIONS
    ],
    batch_size=64
)


del embedding_model

gc.collect()


if torch.cuda.is_available():
    torch.cuda.empty_cache()


TEST_QUERY_EMBEDDINGS = np.asarray(
    TEST_QUERY_EMBEDDINGS,
    dtype=np.float32
)

NOANSWER_QUERY_EMBEDDINGS = np.asarray(
    NOANSWER_QUERY_EMBEDDINGS,
    dtype=np.float32
)


print(
    "TEST:",
    TEST_QUERY_EMBEDDINGS.shape
)

print(
    "NO-ANSWER:",
    NOANSWER_QUERY_EMBEDDINGS.shape
)

NameError: name 'load_embedding_model' is not defined

In [69]:
# ============================================================
# DENSE FAISS
# ============================================================

dense_index = faiss.IndexFlatIP(
    DOCUMENT_EMBEDDINGS.shape[1]
)

dense_index.add(
    DOCUMENT_EMBEDDINGS
)


def dense_rank_questions(
    questions,
    query_embeddings,
    top_k=50,
):

    _, indices = dense_index.search(
        query_embeddings,
        top_k
    )

    output = {}

    for q, row_indices in zip(
        questions,
        indices
    ):

        output[
            q["question_id"]
        ] = [

            FINAL_CHUNKS[
                int(i)
            ]["chunk_id"]

            for i in row_indices

            if i >= 0
        ]

    return output


TEST_DENSE = dense_rank_questions(
    test_questions,
    TEST_QUERY_EMBEDDINGS
)


NOANSWER_DENSE = dense_rank_questions(
    NOANSWER_QUESTIONS,
    NOANSWER_QUERY_EMBEDDINGS
)

NameError: name 'TEST_QUERY_EMBEDDINGS' is not defined

In [70]:
# ============================================================
# BM25
# ============================================================

def bm25_tokenize(text):

    return re.findall(
        r"\w+",
        str(text).lower(),
        flags=re.UNICODE
    )


bm25_corpus = [

    bm25_tokenize(
        c["text"]
    )

    for c in FINAL_CHUNKS
]


FINAL_BM25 = BM25Okapi(
    bm25_corpus
)


def bm25_rank_questions(
    questions,
    top_k=50
):

    output = {}

    for q in questions:

        scores = FINAL_BM25.get_scores(
            bm25_tokenize(
                q["question"]
            )
        )

        order = np.argsort(
            scores
        )[::-1][:top_k]

        output[
            q["question_id"]
        ] = [

            FINAL_CHUNKS[
                int(i)
            ]["chunk_id"]

            for i in order
        ]

    return output


TEST_BM25 = bm25_rank_questions(
    test_questions
)


NOANSWER_BM25 = bm25_rank_questions(
    NOANSWER_QUESTIONS
)

In [71]:
# ============================================================
# RECIPROCAL RANK FUSION
# ============================================================

def rrf_rank(
    dense_ranking,
    bm25_ranking,
    rrf_k=60,
    top_k=50
):

    scores = {}

    for ranking in [
        dense_ranking,
        bm25_ranking
    ]:

        for rank, chunk_id in enumerate(
            ranking,
            start=1
        ):

            scores[
                chunk_id
            ] = (

                scores.get(
                    chunk_id,
                    0.0
                )

                +

                1.0 / (
                    rrf_k + rank
                )
            )

    return [

        chunk_id

        for chunk_id, _
        in sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[:top_k]
    ]


def make_hybrid(
    questions,
    dense,
    bm25
):

    output = {}

    for q in questions:

        qid = q[
            "question_id"
        ]

        output[qid] = rrf_rank(
            dense[qid],
            bm25[qid],
            rrf_k=FINAL_RRF_K,
            top_k=50
        )

    return output


TEST_HYBRID = make_hybrid(
    test_questions,
    TEST_DENSE,
    TEST_BM25
)


NOANSWER_HYBRID = make_hybrid(
    NOANSWER_QUESTIONS,
    NOANSWER_DENSE,
    NOANSWER_BM25
)

NameError: name 'TEST_DENSE' is not defined

In [16]:
# ============================================================
# LOAD FROZEN BGE RERANKER + DEFINE BATCHED RERANK FUNCTION
# ============================================================

import gc
import torch
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)


RERANKER_NAME = "BAAI/bge-reranker-v2-m3"

RERANKER_DEVICE = (
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 80)
print("LOADING FROZEN RERANKER")
print("=" * 80)

print("Model :", RERANKER_NAME)
print("Device:", RERANKER_DEVICE)


# ------------------------------------------------------------
# TOKENIZER
# ------------------------------------------------------------

reranker_tokenizer = AutoTokenizer.from_pretrained(
    RERANKER_NAME
)


# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

reranker_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        RERANKER_NAME,
        torch_dtype=(
            torch.float16
            if RERANKER_DEVICE.startswith("cuda")
            else torch.float32
        ),
    )
)


reranker_model = reranker_model.to(
    RERANKER_DEVICE
)

reranker_model.eval()


print("✅ Reranker loaded")


# ============================================================
# BATCHED RERANK FUNCTION
# ============================================================

def rerank_results_batched(
    questions,
    rankings,
    chunks,
    top_n=30,
    final_k=50,
    batch_size=16,
    max_length=512,
):
    """
    Rerank the first top_n candidates for each question using
    BAAI/bge-reranker-v2-m3.

    rankings:
        dict[question_id] -> list[chunk_id]

    Returns:
        dict[question_id] -> reranked list[chunk_id]

    The top_n candidates are cross-encoder reranked.
    Candidates after top_n preserve their original order.
    """

    # --------------------------------------------------------
    # Chunk lookup
    # --------------------------------------------------------

    chunk_lookup = {
        chunk["chunk_id"]: chunk
        for chunk in chunks
    }


    # --------------------------------------------------------
    # Build all query-passage pairs
    # --------------------------------------------------------

    pair_records = []

    for question in questions:

        qid = question["question_id"]
        query = question["question"]

        candidates = rankings[qid][
            :top_n
        ]

        for chunk_id in candidates:

            if chunk_id not in chunk_lookup:
                raise KeyError(
                    f"Chunk not found: {chunk_id}"
                )

            passage = chunk_lookup[
                chunk_id
            ]["text"]

            pair_records.append({
                "question_id":
                    qid,

                "chunk_id":
                    chunk_id,

                "query":
                    query,

                "passage":
                    passage,
            })


    print(
        f"Reranking {len(pair_records)} "
        f"query-passage pairs..."
    )


    # --------------------------------------------------------
    # Batched inference
    # --------------------------------------------------------

    scores = []


    for start in tqdm(
        range(
            0,
            len(pair_records),
            batch_size
        ),
        desc="Reranker batches",
    ):

        batch = pair_records[
            start:
            start + batch_size
        ]


        pairs = [
            [
                item["query"],
                item["passage"]
            ]
            for item in batch
        ]


        encoded = reranker_tokenizer(
            pairs,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )


        encoded = {
            key: value.to(
                RERANKER_DEVICE
            )
            for key, value
            in encoded.items()
        }


        with torch.inference_mode():

            output = reranker_model(
                **encoded
            )

            logits = (
                output.logits
                .view(-1)
                .float()
                .cpu()
                .numpy()
            )


        scores.extend(
            logits.tolist()
        )


    assert len(scores) == len(
        pair_records
    )


    # --------------------------------------------------------
    # Collect scores by question
    # --------------------------------------------------------

    scores_by_question = {}

    for item, score in zip(
        pair_records,
        scores
    ):

        qid = item[
            "question_id"
        ]

        scores_by_question.setdefault(
            qid,
            []
        )

        scores_by_question[qid].append(
            (
                item["chunk_id"],
                float(score)
            )
        )


    # --------------------------------------------------------
    # Sort top_n by reranker score
    # --------------------------------------------------------

    final_rankings = {}


    for question in questions:

        qid = question[
            "question_id"
        ]

        scored_candidates = (
            scores_by_question[
                qid
            ]
        )


        reranked_top = [
            chunk_id
            for chunk_id, _
            in sorted(
                scored_candidates,
                key=lambda x: x[1],
                reverse=True,
            )
        ]


        # Candidates below top_n remain untouched
        original_tail = rankings[
            qid
        ][top_n:]


        combined = (
            reranked_top
            +
            original_tail
        )


        # Remove accidental duplicates while
        # preserving rank order
        seen = set()
        deduplicated = []

        for chunk_id in combined:

            if chunk_id not in seen:

                seen.add(
                    chunk_id
                )

                deduplicated.append(
                    chunk_id
                )


        final_rankings[qid] = (
            deduplicated[
                :final_k
            ]
        )


    print(
        "✅ Reranking complete"
    )


    return final_rankings

LOADING FROZEN RERANKER
Model : BAAI/bge-reranker-v2-m3
Device: cuda:0


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ Reranker loaded


In [72]:
# ============================================================
# BGE RERANKER — FINAL LOCKED TEST
# ============================================================

TEST_FINAL_RANKINGS = (
    rerank_results_batched(
        questions=test_questions,
        rankings=TEST_HYBRID,
        chunks=FINAL_CHUNKS,
        top_n=30,
        final_k=50,
        batch_size=16,
    )
)


NOANSWER_FINAL_RANKINGS = (
    rerank_results_batched(
        questions=NOANSWER_QUESTIONS,
        rankings=NOANSWER_HYBRID,
        chunks=FINAL_CHUNKS,
        top_n=30,
        final_k=50,
        batch_size=16,
    )
)


print("\n✅ FINAL TEST retrieval complete")
print("✅ NO-ANSWER retrieval complete")

NameError: name 'TEST_HYBRID' is not defined

In [18]:
from pathlib import Path

WORK_DIR = Path(
    "/kaggle/working/docmind_final_v1"
)

WORK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("WORK_DIR:", WORK_DIR)
print("Exists:", WORK_DIR.exists())

print("\nCurrent files:")
for p in WORK_DIR.iterdir():
    print(" -", p.name)

WORK_DIR: /kaggle/working/docmind_final_v1
Exists: True

Current files:


In [130]:
# ============================================================
# GOLD EVIDENCE MATCHING + FINAL RETRIEVAL METRICS
# ============================================================

def evidence_matches_chunk(evidence, chunk):
    """
    Return True when a chunk satisfies one gold evidence unit.

    Supports the gold format used in DocMind:
    - document_id / doc_id
    - location_type
    - location_value
    - contains / anchors
    """

    # --------------------------------------------------------
    # 1. DOCUMENT MATCH
    # --------------------------------------------------------

    evidence_doc = (
        evidence.get("document_id")
        or evidence.get("doc_id")
    )

    chunk_doc = (
        chunk.get("doc_id")
        or chunk.get("document_id")
    )

    if evidence_doc is not None:
        if chunk_doc != evidence_doc:
            return False


    # --------------------------------------------------------
    # 2. LOCATION TYPE MATCH
    # page / slide / etc.
    # --------------------------------------------------------

    evidence_location_type = evidence.get(
        "location_type"
    )

    if evidence_location_type is not None:

        if (
            str(chunk.get("location_type")).lower()
            !=
            str(evidence_location_type).lower()
        ):
            return False


    # --------------------------------------------------------
    # 3. LOCATION VALUE MATCH
    # --------------------------------------------------------

    evidence_location_value = evidence.get(
        "location_value"
    )

    # Some gold records may instead store page/slide directly.
    if evidence_location_value is None:

        if "page" in evidence:
            evidence_location_value = evidence["page"]

        elif "slide" in evidence:
            evidence_location_value = evidence["slide"]


    if evidence_location_value is not None:

        if str(
            chunk.get("location_value")
        ) != str(
            evidence_location_value
        ):
            return False


    # --------------------------------------------------------
    # 4. TEXT / ANCHOR MATCH
    # --------------------------------------------------------

    chunk_text = str(
        chunk.get("text", "")
    ).lower()


    # DocMind gold may use "contains"
    contains = evidence.get(
        "contains"
    )

    # Or anchors
    if contains is None:
        contains = evidence.get(
            "anchors"
        )


    # No anchor requirement -> location match is enough
    if not contains:
        return True


    # Normalize single string -> list
    if isinstance(
        contains,
        str
    ):
        contains = [contains]


    # All anchors belonging to THIS evidence unit
    # must occur in the chunk.
    #
    # q057 was already fixed by splitting
    # Volume / Vitesse / Variété into separate
    # evidence units, so this behavior is correct.
    return all(
        str(anchor).lower()
        in chunk_text

        for anchor in contains
    )


# ============================================================
# BUILD GOLD EVIDENCE GROUPS
# ============================================================

def build_gold_evidence_groups(
    questions,
    chunks
):
    """
    Map each gold evidence unit to every chunk that can satisfy it.

    Example:
        q057:
            Evidence unit 1 -> chunks containing Volume
            Evidence unit 2 -> chunks containing Vitesse
            Evidence unit 3 -> chunks containing Variété
    """

    result = {}


    for q in questions:

        qid = q["question_id"]

        groups = []


        for evidence in q.get(
            "gold_evidence",
            []
        ):

            matching_chunks = {

                chunk["chunk_id"]

                for chunk in chunks

                if evidence_matches_chunk(
                    evidence,
                    chunk
                )
            }


            groups.append(
                matching_chunks
            )


        result[qid] = groups


    return result


# ============================================================
# EVIDENCE RECALL @ K
# ============================================================

def evidence_recall_at_k(
    ranking,
    evidence_groups,
    k
):
    """
    Fraction of required evidence units represented
    within the top-k retrieved chunks.
    """

    if not evidence_groups:
        return 0.0


    retrieved = set(
        ranking[:k]
    )


    covered = sum(

        bool(
            retrieved & group
        )

        for group
        in evidence_groups
    )


    return (
        covered
        /
        len(evidence_groups)
    )


# ============================================================
# COMPLETE EVIDENCE @ K
# ============================================================

def complete_evidence_at_k(
    ranking,
    evidence_groups,
    k
):
    """
    1 if EVERY required evidence unit is represented
    in the top-k retrieval results, otherwise 0.
    """

    if not evidence_groups:
        return 0.0


    retrieved = set(
        ranking[:k]
    )


    return float(

        all(

            bool(
                retrieved & group
            )

            for group
            in evidence_groups
        )
    )


print("✅ Evidence matching helpers loaded")
print("✅ build_gold_evidence_groups loaded")
print("✅ EvidenceRecall@K loaded")
print("✅ CompleteEvidence@K loaded")

✅ Evidence matching helpers loaded
✅ build_gold_evidence_groups loaded
✅ EvidenceRecall@K loaded
✅ CompleteEvidence@K loaded


In [131]:
# ============================================================
# VERIFY TEST GOLD EVIDENCE CAN BE FOUND IN MEDIUM CHUNKS
# ============================================================

TEST_EVIDENCE_GROUPS = (
    build_gold_evidence_groups(
        test_questions,
        FINAL_CHUNKS
    )
)


problems = []


for q in test_questions:

    qid = q["question_id"]

    groups = TEST_EVIDENCE_GROUPS[
        qid
    ]

    for i, group in enumerate(
        groups,
        start=1
    ):

        if len(group) == 0:

            problems.append({
                "question_id":
                    qid,

                "evidence_unit":
                    i,

                "question":
                    q["question"]
            })


if problems:

    print(
        "❌ GOLD COVERAGE PROBLEM"
    )

    display(
        pd.DataFrame(problems)
    )

else:

    print(
        "✅ TEST gold evidence coverage is complete"
    )

    print(
        "Questions:",
        len(test_questions)
    )

    print(
        "All required evidence units map "
        "to at least one medium chunk."
    )

✅ TEST gold evidence coverage is complete
Questions: 12
All required evidence units map to at least one medium chunk.


In [132]:
# ============================================================
# EXPORT TEST + NO-ANSWER FROZEN CONTEXTS
# ============================================================

CHUNK_LOOKUP = {
    c["chunk_id"]: c
    for c in FINAL_CHUNKS
}


def build_generation_records(
    questions,
    rankings
):

    output = []

    for q in questions:

        qid = q["question_id"]

        top_ids = rankings[qid][
            :FINAL_TOP_K
        ]

        contexts = []

        for rank, chunk_id in enumerate(
            top_ids,
            start=1
        ):

            c = CHUNK_LOOKUP[
                chunk_id
            ]

            contexts.append({
                "source_id":
                    f"S{rank}",

                "rank":
                    rank,

                "chunk_id":
                    chunk_id,

                "document_id":
                    c["doc_id"],

                "location_type":
                    c["location_type"],

                "location_value":
                    c["location_value"],

                "text":
                    c["text"],
            })


        output.append({
            "question_id":
                qid,

            "question":
                q["question"],

            "language":
                q["language"],

            "question_type":
                q["question_type"],

            "answerable":
                q["answerable"],

            "expected_answer":
                q.get(
                    "expected_answer"
                ),

            "contexts":
                contexts,
        })


    return output


TEST_GENERATION_RECORDS = (
    build_generation_records(
        test_questions,
        TEST_FINAL_RANKINGS
    )
)


NOANSWER_GENERATION_RECORDS = (
    build_generation_records(
        NOANSWER_QUESTIONS,
        NOANSWER_FINAL_RANKINGS
    )
)


FROZEN_CONFIG = {
    "embedding":
        "Qwen/Qwen3-Embedding-0.6B",

    "chunk_size_tokens":
        320,

    "chunk_overlap_tokens":
        48,

    "retrieval":
        "dense + BM25",

    "fusion":
        "RRF",

    "rrf_k":
        60,

    "reranker":
        "BAAI/bge-reranker-v2-m3",

    "rerank_top_n":
        30,

    "final_k":
        5,
}


TEST_JSON_PATH = (
    WORK_DIR /
    "docmind_generation_test_frozen.json"
)


NOANSWER_JSON_PATH = (
    WORK_DIR /
    "docmind_generation_noanswer_frozen.json"
)


with open(
    TEST_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "split":
                "TEST",

            "frozen":
                True,

            "retrieval_config":
                FROZEN_CONFIG,

            "records":
                TEST_GENERATION_RECORDS,
        },

        f,

        ensure_ascii=False,
        indent=2,
    )


with open(
    NOANSWER_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "split":
                "NOANSWER",

            "frozen":
                True,

            "retrieval_config":
                FROZEN_CONFIG,

            "records":
                NOANSWER_GENERATION_RECORDS,
        },

        f,

        ensure_ascii=False,
        indent=2,
    )


print("✅ Saved:")
print(TEST_JSON_PATH)
print(NOANSWER_JSON_PATH)

print(
    "\nTEST records:",
    len(TEST_GENERATION_RECORDS)
)

print(
    "NO-ANSWER records:",
    len(NOANSWER_GENERATION_RECORDS)
)

✅ Saved:
/kaggle/working/docmind_final_v1/docmind_generation_test_frozen.json
/kaggle/working/docmind_final_v1/docmind_generation_noanswer_frozen.json

TEST records: 12
NO-ANSWER records: 4


In [133]:
import shutil

# Compress /kaggle/working/docmind_research into docmind_research.zip
shutil.make_archive('docmind_final_v1', 'zip', '/kaggle/working/docmind_final_v1')

'/kaggle/working/docmind_final_v1.zip'

In [2]:
# Cell 1 — Install dependencies
!pip install -q \
    pymupdf \
    python-docx \
    python-pptx \
    sentence-transformers \
    transformers \
    faiss-cpu \
    rank-bm25 \
    scikit-learn \
    langchain-text-splitters \
    psutil \
    tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 62.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 79.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 11.9 MB/s eta 0:00:00


In [3]:
# Cell 2 — Imports and experiment configuration
import os
import re
import gc
import json
import time
import math
import random
import hashlib
import platform
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

import fitz
from docx import Document
from pptx import Presentation

from tqdm.auto import tqdm
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter

from rank_bm25 import BM25Okapi
import faiss

from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("/kaggle/input/datasets/ripperdzz/docmind-test")
WORK_DIR = Path("/kaggle/working/docmind_research")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR :", DATA_DIR)
print("WORK_DIR :", WORK_DIR)
print("CPU cores:", os.cpu_count())
print("RAM GB   :", round(psutil.virtual_memory().total / 1024**3, 2))

DATA_DIR : /kaggle/input/datasets/ripperdzz/docmind-test
WORK_DIR : /kaggle/working/docmind_research
CPU cores: 4
RAM GB   : 31.35


In [4]:
EXPECTED_FILES = ['1. Introduction.pdf', '10. HMM.pdf', '11. DTW.pdf', '2. Concepts fondamentaux.pdf', '3. Regression.pdf', '4. Regression Logistique.pdf', '5. Non-paramtriques-KNN.pdf', '6. Naive Bayes.pdf', '7. Clustering - part2.pdf', '7. Clustering-part1.pdf', '8. ANN-part1.pdf', '8. ANN-part2.pdf', '9. SVM.pdf', 'AI and Biases.pptx', 'Academic writing.pptx', 'An introduction to Cognitive Science.pptx', 'Artificial Intelligence - An Overview.pptx', 'BMC I2E final1 (1).docx', 'Cc_Logique_Mathematique_2015-2016.docx', 'Ch1_Algbre1-2021-2022.pdf', 'Ch2_Algbre1-2021-2022.pdf', 'Chap5_cours_Analyse1.pdf', 'Cloud Computing Enhancing AI.pptx', 'Cours_Data_ScienceComplet.pdf', 'Data Science Examen de Remplacement.pdf', 'Ethical AI.pptx', 'Examen Final (Corrig).pdf', 'Intelligence Theory and Brain Properties.pptx', 'L1_MI_TD0_analyse2_21.pdf', 'L1_MI_TD1_analyse2_21.pdf', 'L1_MI_TD2_Analyse2_21.pdf', 'Machine Learning.pptx', 'RNN.pptx', 'Rattrapage Data Science IA.pdf', 'Software Processes and Agile Practices.pdf', 'Solution Data Science Examen (1) (1).pdf', 'Sujet_corrige_CC_Analyse-1_21-22.pdf', 'TD N2-DM-25-26.pdf', 'TP-N2-2023-2024.docx', 'Transformers.pptx', 'controle-systeme-1-2019.docx', 'controle-systeme-2020.docx', 'controle-tp-finale-2016.docx', 'frequents-sequence-2026-P1.pdf']

print("Expected corpus files:", len(EXPECTED_FILES))


Expected corpus files: 44


In [5]:
# Cell 4 — Discover files and enforce the clean corpus allow-list

all_found = []
for root, _, filenames in os.walk(DATA_DIR):
    for name in filenames:
        all_found.append(Path(root) / name)

found_by_name = {p.name: p for p in all_found}

missing = [name for name in EXPECTED_FILES if name not in found_by_name]
extra = [name for name in found_by_name if name not in set(EXPECTED_FILES)]

print(f"Found in Kaggle dataset : {len(all_found)}")
print(f"Expected clean corpus   : {len(EXPECTED_FILES)}")
print(f"Missing expected files  : {len(missing)}")
print(f"Extra ignored files     : {len(extra)}")

if missing:
    print("\nMISSING:")
    for name in missing:
        print(" -", name)

if extra:
    print("\nIGNORED EXTRAS:")
    for name in extra[:30]:
        print(" -", name)

FILES = [found_by_name[name] for name in EXPECTED_FILES if name in found_by_name]

print("\nFiles that will actually be benchmarked:", len(FILES))

Found in Kaggle dataset : 44
Expected clean corpus   : 44
Missing expected files  : 0
Extra ignored files     : 0

Files that will actually be benchmarked: 44


In [6]:
# Cell 5 — Text normalization

CONTROL_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

def clean_text(text: str) -> str:
    if text is None:
        return ""

    text = str(text)
    text = text.replace("\xa0", " ")
    text = CONTROL_CHARS.sub(" ", text)

    # Preserve paragraph boundaries while normalizing noisy spacing.
    lines = []
    for line in text.splitlines():
        line = re.sub(r"[ \t]+", " ", line).strip()
        if line:
            lines.append(line)

    return "\n".join(lines).strip()


def text_hash(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()

In [7]:
# Cell 6 — PDF extraction: one page = one source block

def extract_pdf(path: Path):
    blocks = []
    doc = fitz.open(path)

    try:
        for page_number, page in enumerate(doc, start=1):
            text = clean_text(page.get_text("text"))

            if not text:
                continue

            blocks.append({
                "doc_id": path.name,
                "file_type": "pdf",
                "block_type": "page",
                "location_type": "page",
                "location_value": page_number,
                "text": text,
                "char_count": len(text),
                "text_hash": text_hash(text),
            })
    finally:
        doc.close()

    return blocks

In [8]:
# Cell 7 — DOCX extraction
# Paragraphs are grouped into a document-text block.
# Tables are extracted separately so table text does not get silently lost.

def extract_docx(path: Path):
    blocks = []
    doc = Document(path)

    paragraphs = [
        clean_text(p.text)
        for p in doc.paragraphs
        if clean_text(p.text)
    ]

    if paragraphs:
        text = "\n".join(paragraphs)
        blocks.append({
            "doc_id": path.name,
            "file_type": "docx",
            "block_type": "document_text",
            "location_type": "document",
            "location_value": 1,
            "text": text,
            "char_count": len(text),
            "text_hash": text_hash(text),
        })

    for table_index, table in enumerate(doc.tables, start=1):
        rows = []

        for row in table.rows:
            values = [clean_text(cell.text) for cell in row.cells]
            if any(values):
                rows.append(" | ".join(values))

        if not rows:
            continue

        text = "\n".join(rows)

        blocks.append({
            "doc_id": path.name,
            "file_type": "docx",
            "block_type": "table",
            "location_type": "table",
            "location_value": table_index,
            "text": text,
            "char_count": len(text),
            "text_hash": text_hash(text),
        })

    return blocks

In [9]:
# Cell 8 — PPTX extraction
# Keep each slide as its own retrieval location.
# Include visible shape text and table cells.

def extract_pptx(path: Path):
    blocks = []
    prs = Presentation(path)

    for slide_number, slide in enumerate(prs.slides, start=1):
        parts = []

        for shape in slide.shapes:
            # Normal text boxes / titles
            if hasattr(shape, "text"):
                text = clean_text(shape.text)
                if text:
                    parts.append(text)

            # Tables
            if getattr(shape, "has_table", False):
                for row in shape.table.rows:
                    values = [clean_text(cell.text) for cell in row.cells]
                    if any(values):
                        parts.append(" | ".join(values))

        text = clean_text("\n".join(parts))

        if not text:
            continue

        blocks.append({
            "doc_id": path.name,
            "file_type": "pptx",
            "block_type": "slide",
            "location_type": "slide",
            "location_value": slide_number,
            "text": text,
            "char_count": len(text),
            "text_hash": text_hash(text),
        })

    return blocks

In [10]:
# Cell 9 — Run extraction over the clean corpus

EXTRACTORS = {
    ".pdf": extract_pdf,
    ".docx": extract_docx,
    ".pptx": extract_pptx,
}

normalized_blocks = []
extraction_errors = []

for path in tqdm(FILES, desc="Extracting"):
    ext = path.suffix.lower()

    if ext not in EXTRACTORS:
        extraction_errors.append({
            "doc_id": path.name,
            "error": f"Unsupported extension: {ext}",
        })
        continue

    try:
        normalized_blocks.extend(EXTRACTORS[ext](path))
    except Exception as exc:
        extraction_errors.append({
            "doc_id": path.name,
            "error": repr(exc),
        })

print("Documents attempted :", len(FILES))
print("Extracted blocks     :", len(normalized_blocks))
print("Extraction errors    :", len(extraction_errors))

if extraction_errors:
    pd.DataFrame(extraction_errors)

Extracting:   0%|          | 0/44 [00:00<?, ?it/s]

Documents attempted : 44
Extracted blocks     : 935
Extraction errors    : 0


In [11]:
# Cell 10 — Extraction audit by document

audit_rows = []

by_doc = defaultdict(list)
for block in normalized_blocks:
    by_doc[block["doc_id"]].append(block)

for doc_id in sorted(set(p.name for p in FILES)):
    blocks = by_doc.get(doc_id, [])

    char_counts = [b["char_count"] for b in blocks]
    hashes = [b["text_hash"] for b in blocks]

    audit_rows.append({
        "doc_id": doc_id,
        "blocks": len(blocks),
        "total_chars": sum(char_counts),
        "median_chars_per_block": float(np.median(char_counts)) if char_counts else 0,
        "tiny_blocks_lt_30_chars": sum(n < 30 for n in char_counts),
        "tiny_block_rate": (
            sum(n < 30 for n in char_counts) / len(char_counts)
            if char_counts else 1.0
        ),
        "duplicate_block_count": len(hashes) - len(set(hashes)),
        "empty_extraction": len(blocks) == 0,
    })

audit_df = pd.DataFrame(audit_rows).sort_values(
    ["empty_extraction", "tiny_block_rate", "total_chars"],
    ascending=[False, False, True],
)

pd.set_option("display.max_rows", 100)
display(audit_df)

print("\nProblem candidates:")
display(
    audit_df[
        (audit_df["empty_extraction"]) |
        (audit_df["tiny_block_rate"] > 0.35)
    ]
)

,doc_id,blocks,total_chars,median_chars_per_block,tiny_blocks_lt_30_chars,tiny_block_rate,duplicate_block_count,empty_extraction
28,L1_MI_TD0_analyse2_21.pdf,0,0,0.0,0,1.000000,0,True
31,Machine Learning.pptx,16,5226,352.0,6,0.375000,0,False
13,AI and Biases.pptx,15,4511,379.0,4,0.266667,1,False
16,Artificial Intelligence - An Overview.pptx,13,4943,360.0,3,0.230769,0,False
20,Ch2_Algbre1-2021-2022.pdf,10,6874,763.0,2,0.200000,0,False
27,Intelligence Theory and Brain Properties.pptx,11,3056,234.0,2,0.181818,0,False
22,Cloud Computing Enhancing AI.pptx,14,4669,365.0,2,0.142857,0,False
25,Ethical AI.pptx,14,4695,328.0,2,0.142857,0,False
8,7. Clustering - part2.pdf,63,16250,181.0,7,0.111111,0,False
39,Transformers.pptx,19,3972,200.0,2,0.105263,0,False



Problem candidates:


,doc_id,blocks,total_chars,median_chars_per_block,tiny_blocks_lt_30_chars,tiny_block_rate,duplicate_block_count,empty_extraction
28,L1_MI_TD0_analyse2_21.pdf,0,0,0.0,0,1.000,0,True
31,Machine Learning.pptx,16,5226,352.0,6,0.375,0,False


In [43]:
# Cell 11 — Inspect representative extracted blocks

def show_document(doc_id, max_blocks=5, max_chars=2500):
    blocks = [b for b in normalized_blocks if b["doc_id"] == doc_id]

    print(f"{doc_id} — {len(blocks)} blocks")

    for block in blocks[:max_blocks]:
        print("\n" + "=" * 100)
        print(
            block["block_type"],
            "|",
            block["location_type"],
            block["location_value"]
        )
        print("-" * 100)
        print(block["text"][:max_chars])


# Change these names when you want to inspect another file.
for sample_doc in [
    "3. Regression.pdf",
    "8. ANN-part2.pdf",
    "Machine Learning.pptx",
    "BMC I2E final1 (1).docx",
]:
    if sample_doc in by_doc:
        show_document(sample_doc, max_blocks=2)

3. Regression.pdf — 42 blocks

page | page 1
----------------------------------------------------------------------------------------------------
R E G R E S S I O N
▪Régression linéaire simple
▪Régression linéaire multiple
▪Régression polynomiale

page | page 2
----------------------------------------------------------------------------------------------------
Introduction
2
Température,
Âge, Salaire
8. ANN-part2.pdf — 61 blocks

page | page 1
----------------------------------------------------------------------------------------------------
MLP : MultiLayer Perceptron

page | page 2
----------------------------------------------------------------------------------------------------
TREY
Research
2
Limitations du Perceptron
L’exemple historique du XOR
▪
Le perceptron permet de séparer les classes linéairement séparables, mais il est incapable de séparer
des classes non linéaires.
▪
Le problème du XOR de classification binaire (1969) avec l’opération booléenne qui correspond au
non ex

In [44]:
# Cell 12 — Export extraction snapshot

EXTRACTION_PATH = WORK_DIR / "docmind_clean_extracted_blocks.json"
AUDIT_PATH = WORK_DIR / "docmind_extraction_audit.csv"

with open(EXTRACTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        normalized_blocks,
        f,
        ensure_ascii=False,
        indent=2,
    )

audit_df.to_csv(AUDIT_PATH, index=False)

print("Saved:", EXTRACTION_PATH)
print("Saved:", AUDIT_PATH)

Saved: /kaggle/working/docmind_final_v1/docmind_clean_extracted_blocks.json
Saved: /kaggle/working/docmind_final_v1/docmind_extraction_audit.csv


## STOP 1 — Extraction gate

Before benchmarking embeddings:

1. Read the extraction audit.
2. Inspect several PDFs, PPTX files, and DOCX files.
3. Do not create gold questions whose answers are only visible in figures/equations that the text extractor lost.
4. If a document has very poor extraction, flag it as an extraction-stress document instead of blaming retrieval.

Once this looks acceptable, continue to chunking.

In [45]:
# Cell 14 — Token-aware chunking configuration
#
# We use one tokenizer for the chunk-size experiment so all models receive
# the same chunk boundaries. The embedding model itself is NOT being chosen here.

CHUNK_TOKENIZER_NAME = "intfloat/multilingual-e5-large"
chunk_tokenizer = AutoTokenizer.from_pretrained(CHUNK_TOKENIZER_NAME)

def token_length(text: str) -> int:
    return len(
        chunk_tokenizer.encode(
            text,
            add_special_tokens=False,
            truncation=False,
        )
    )

CHUNK_CONFIGS = {
    "small":  {"chunk_tokens": 192, "overlap_tokens": 32},
    "medium": {"chunk_tokens": 320, "overlap_tokens": 48},
    "large":  {"chunk_tokens": 448, "overlap_tokens": 64},
}

CHUNK_CONFIGS

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

{'small': {'chunk_tokens': 192, 'overlap_tokens': 32},
 'medium': {'chunk_tokens': 320, 'overlap_tokens': 48},
 'large': {'chunk_tokens': 448, 'overlap_tokens': 64}}

In [46]:
# Cell 15 — Build chunks while preserving source metadata

def build_chunks(blocks, config_name):
    cfg = CHUNK_CONFIGS[config_name]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg["chunk_tokens"],
        chunk_overlap=cfg["overlap_tokens"],
        length_function=token_length,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = []

    for block_index, block in enumerate(tqdm(blocks, desc=f"Chunking {config_name}")):
        parts = splitter.split_text(block["text"])

        for part_index, text in enumerate(parts):
            text = clean_text(text)

            if not text:
                continue

            chunk_id = (
                f"{config_name}|{block['doc_id']}|"
                f"{block['location_type']}:{block['location_value']}|"
                f"{part_index}"
            )

            chunks.append({
                "chunk_id": chunk_id,
                "chunk_config": config_name,
                "doc_id": block["doc_id"],
                "file_type": block["file_type"],
                "block_type": block["block_type"],
                "location_type": block["location_type"],
                "location_value": block["location_value"],
                "part_index": part_index,
                "text": text,
                "token_count": token_length(text),
            })

    return chunks


chunks_by_config = {
    name: build_chunks(normalized_blocks, name)
    for name in CHUNK_CONFIGS
}

for name, chunks in chunks_by_config.items():
    token_counts = [c["token_count"] for c in chunks]

    print(
        name,
        "| chunks =", len(chunks),
        "| median tokens =", round(float(np.median(token_counts)), 1),
        "| p95 =", round(float(np.percentile(token_counts, 95)), 1),
        "| max =", max(token_counts),
    )

Chunking small:   0%|          | 0/935 [00:00<?, ?it/s]

Chunking medium:   0%|          | 0/935 [00:00<?, ?it/s]

Chunking large:   0%|          | 0/935 [00:00<?, ?it/s]

small | chunks = 1546 | median tokens = 140.0 | p95 = 192.0 | max = 192
medium | chunks = 1210 | median tokens = 125.0 | p95 = 319.0 | max = 320
large | chunks = 1092 | median tokens = 118.0 | p95 = 445.0 | max = 448


In [47]:
# Cell 16 — Export all chunk configurations

for name, chunks in chunks_by_config.items():
    path = WORK_DIR / f"docmind_chunks_{name}.json"

    with open(path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    print("Saved:", path)

Saved: /kaggle/working/docmind_final_v1/docmind_chunks_small.json
Saved: /kaggle/working/docmind_final_v1/docmind_chunks_medium.json
Saved: /kaggle/working/docmind_final_v1/docmind_chunks_large.json


## Gold benchmark format

Use a JSON file, not CSV.

Each question should identify the exact evidence location(s) needed to answer it.

Recommended question types:

- `fact`
- `definition`
- `procedure`
- `conceptual`
- `comparison`
- `multi_document`
- `cross_format`
- `no_answer`

Recommended languages:

- `en`
- `fr`
- `ar`
- `mixed`

Do **not** invent page/slide evidence. Verify it against the extraction first.

In [14]:
# Cell 18 — Load / create gold-question file

GOLD_PATH = WORK_DIR / "rag_gold_questions.json"

if not GOLD_PATH.exists():
    template = {
        "schema_version": "1.0",
        "questions": []
    }

    with open(GOLD_PATH, "w", encoding="utf-8") as f:
        json.dump(template, f, ensure_ascii=False, indent=2)

    print("Created empty gold file:", GOLD_PATH)
    print("Populate it only with manually verified questions/evidence.")

with open(GOLD_PATH, "r", encoding="utf-8") as f:
    gold_data = json.load(f)

gold_questions = gold_data["questions"]

print("Gold questions:", len(gold_questions))

Created empty gold file: /kaggle/working/docmind_research/rag_gold_questions.json
Populate it only with manually verified questions/evidence.
Gold questions: 0


In [49]:
# Cell 19 — Gold schema validator

ALLOWED_LANGUAGES = {"en", "fr", "ar", "mixed"}

ALLOWED_TYPES = {
    "fact",
    "definition",
    "procedure",
    "conceptual",
    "comparison",
    "multi_document",
    "cross_format",
    "no_answer",
}

def validate_gold_questions(questions):
    errors = []
    seen_ids = set()
    valid_docs = set(p.name for p in FILES)

    for i, q in enumerate(questions):
        prefix = f"questions[{i}]"

        qid = q.get("question_id")
        if not qid:
            errors.append(f"{prefix}: missing question_id")
        elif qid in seen_ids:
            errors.append(f"{prefix}: duplicate question_id {qid}")
        else:
            seen_ids.add(qid)

        if not q.get("question"):
            errors.append(f"{prefix}: missing question")

        if q.get("language") not in ALLOWED_LANGUAGES:
            errors.append(f"{prefix}: invalid language")

        if q.get("question_type") not in ALLOWED_TYPES:
            errors.append(f"{prefix}: invalid question_type")

        answerable = q.get("answerable")
        evidence = q.get("gold_evidence", [])

        if answerable is True and not evidence:
            errors.append(f"{prefix}: answerable question has no gold_evidence")

        if answerable is False and evidence:
            errors.append(f"{prefix}: no-answer question should have empty gold_evidence")

        for j, ev in enumerate(evidence):
            doc_id = ev.get("document_id")
            if doc_id not in valid_docs:
                errors.append(
                    f"{prefix}.gold_evidence[{j}]: unknown document_id {doc_id}"
                )

            loc = ev.get("location", {})
            if "type" not in loc or "value" not in loc:
                errors.append(
                    f"{prefix}.gold_evidence[{j}]: invalid location"
                )

    return errors


gold_errors = validate_gold_questions(gold_questions)

print("Validation errors:", len(gold_errors))
for error in gold_errors[:50]:
    print(" -", error)

Validation errors: 0


In [50]:
# Cell 20 — Match gold evidence to chunks
#
# 'contains' is used only as a deterministic annotation aid:
# all listed strings must appear in the candidate chunk.
# This avoids judging retrieval only by document name.

def normalize_for_match(text):
    return re.sub(r"\s+", " ", clean_text(text)).strip().lower()


def evidence_matches_chunk(evidence, chunk):
    if evidence["document_id"] != chunk["doc_id"]:
        return False

    location = evidence.get("location", {})
    loc_type = location.get("type")
    loc_value = location.get("value")

    if loc_type and chunk["location_type"] != loc_type:
        return False

    if loc_value is not None and chunk["location_value"] != loc_value:
        return False

    required = evidence.get("contains", [])
    chunk_text = normalize_for_match(chunk["text"])

    return all(
        normalize_for_match(term) in chunk_text
        for term in required
    )


def build_gold_chunk_map(questions, chunks):
    result = {}

    for q in questions:
        qid = q["question_id"]

        if not q.get("answerable", False):
            result[qid] = set()
            continue

        matched = set()

        for evidence in q.get("gold_evidence", []):
            for chunk in chunks:
                if evidence_matches_chunk(evidence, chunk):
                    matched.add(chunk["chunk_id"])

        result[qid] = matched

    return result


for config_name, chunks in chunks_by_config.items():
    gold_map = build_gold_chunk_map(gold_questions, chunks)

    answerable = [
        q for q in gold_questions
        if q.get("answerable", False)
    ]

    covered = sum(
        bool(gold_map[q["question_id"]])
        for q in answerable
    )

    coverage = covered / len(answerable) if answerable else 0

    print(
        config_name,
        "| gold evidence coverage =",
        f"{coverage:.3f}",
        f"({covered}/{len(answerable)})"
    )

small | gold evidence coverage = 0.000 (0/0)
medium | gold evidence coverage = 0.000 (0/0)
large | gold evidence coverage = 0.000 (0/0)


## STOP 2 — Gold evidence coverage

Do not compare embedding models until gold evidence coverage is high.

A retrieval model cannot retrieve evidence that extraction/chunking failed to preserve.

A good target is **close to 100% coverage** for manually verified answerable questions.

In [51]:
# Cell 22 — Fixed development / test split

answerable_questions = [
    q for q in gold_questions
    if q.get("answerable", False)
]

if len(answerable_questions) >= 20:
    labels = [
        f"{q['language']}|{q['question_type']}"
        for q in answerable_questions
    ]

    # Stratify only when every stratum has enough examples.
    counts = Counter(labels)
    can_stratify = all(v >= 2 for v in counts.values())

    dev_questions, test_questions = train_test_split(
        answerable_questions,
        test_size=0.20,
        random_state=SEED,
        stratify=labels if can_stratify else None,
    )

    print("DEV :", len(dev_questions))
    print("TEST:", len(test_questions))
else:
    dev_questions = answerable_questions
    test_questions = []
    print(
        "Need at least ~20 answerable questions before making the held-out split."
    )

Need at least ~20 answerable questions before making the held-out split.


In [52]:
# ============================================================
# DIAGNOSE MISSING GOLD EVIDENCE
# ============================================================

for config_name, chunks in chunks_by_config.items():

    gold_map = build_gold_chunk_map(
        answerable_questions,
        chunks
    )

    missing = [
        q
        for q in answerable_questions
        if not gold_map[q["question_id"]]
    ]

    print("\n" + "=" * 100)
    print(
        config_name.upper(),
        f"— missing {len(missing)}"
    )
    print("=" * 100)

    for q in missing:

        print("\nQUESTION ID :", q["question_id"])
        print("QUESTION    :", q["question"])

        for evidence in q["gold_evidence"]:

            print("\nDOCUMENT :", evidence["document_id"])
            print("LOCATION :", evidence["location"])
            print("CONTAINS :", evidence.get("contains"))

            candidates = [
                c
                for c in chunks
                if (
                    c["doc_id"] == evidence["document_id"]
                    and
                    c["location_type"]
                    == evidence["location"]["type"]
                    and
                    c["location_value"]
                    == evidence["location"]["value"]
                )
            ]

            print(
                "\nChunks from the correct location:",
                len(candidates)
            )

            for c in candidates:

                print("\n---", c["chunk_id"], "---")
                print(c["text"])


SMALL — missing 0

MEDIUM — missing 0

LARGE — missing 0


### Multi-passage gold evidence

Question `q057` requires three pieces of evidence: **Volume, Vitesse, and Variété**.

For smaller chunk sizes these facts are distributed across multiple chunks, while the large configuration keeps them together.

Therefore, q057 is represented using multiple gold evidence entries rather than requiring all three terms to occur inside one chunk.

This lets the benchmark measure evidence fragmentation correctly:

- small chunks may need to retrieve 3 relevant chunks;
- medium chunks may need 2 relevant chunks;
- large chunks may need only 1 relevant chunk.

We do not relax the gold matcher because retrieving only one of the three V's is not sufficient evidence for answering the full question.

In [53]:
# ============================================================
# FIX q057 — MULTI-PASSAGE GOLD EVIDENCE
# ============================================================

qid = "q057"

q057 = next(
    q for q in gold_questions
    if q["question_id"] == qid
)

q057["gold_evidence"] = [

    {
        "document_id": "Cours_Data_ScienceComplet.pdf",
        "location": {
            "type": "page",
            "value": 109
        },
        "contains": [
            "Volume"
        ]
    },

    {
        "document_id": "Cours_Data_ScienceComplet.pdf",
        "location": {
            "type": "page",
            "value": 109
        },
        "contains": [
            "Vitesse"
        ]
    },

    {
        "document_id": "Cours_Data_ScienceComplet.pdf",
        "location": {
            "type": "page",
            "value": 109
        },
        "contains": [
            "Variété"
        ]
    }
]


# Save corrected benchmark
gold_data["questions"] = gold_questions

with open(
    GOLD_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        gold_data,
        f,
        ensure_ascii=False,
        indent=2
    )


print("✅ q057 converted to multi-passage evidence")
print(json.dumps(
    q057,
    ensure_ascii=False,
    indent=2
))

StopIteration: 

In [55]:
# ============================================================
# SAVE ONLY NO-ANSWER FROZEN CONTEXTS
# ============================================================

from pathlib import Path
import json

WORK_DIR = Path(
    "/kaggle/working/docmind_final_v1"
)

WORK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHUNK_LOOKUP = {
    c["chunk_id"]: c
    for c in FINAL_CHUNKS
}


def build_generation_records(
    questions,
    rankings
):
    output = []

    for q in questions:

        qid = q["question_id"]

        top_ids = rankings[qid][
            :FINAL_TOP_K
        ]

        contexts = []

        for rank, chunk_id in enumerate(
            top_ids,
            start=1
        ):

            c = CHUNK_LOOKUP[
                chunk_id
            ]

            contexts.append({
                "source_id":
                    f"S{rank}",

                "rank":
                    rank,

                "chunk_id":
                    chunk_id,

                "document_id":
                    c["doc_id"],

                "location_type":
                    c["location_type"],

                "location_value":
                    c["location_value"],

                "text":
                    c["text"],
            })

        output.append({
            "question_id":
                qid,

            "question":
                q["question"],

            "language":
                q["language"],

            "question_type":
                q["question_type"],

            "answerable":
                q["answerable"],

            "expected_answer":
                q.get(
                    "expected_answer"
                ),

            "contexts":
                contexts,
        })

    return output


NOANSWER_GENERATION_RECORDS = (
    build_generation_records(
        NOANSWER_QUESTIONS,
        NOANSWER_FINAL_RANKINGS
    )
)


FROZEN_CONFIG = {
    "embedding":
        "Qwen/Qwen3-Embedding-0.6B",

    "chunk_size_tokens":
        320,

    "chunk_overlap_tokens":
        48,

    "retrieval":
        "dense + BM25",

    "fusion":
        "RRF",

    "rrf_k":
        60,

    "reranker":
        "BAAI/bge-reranker-v2-m3",

    "rerank_top_n":
        30,

    "final_k":
        5,
}


NOANSWER_JSON_PATH = (
    WORK_DIR /
    "docmind_generation_noanswer_frozen.json"
)


with open(
    NOANSWER_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "split":
                "NOANSWER",

            "frozen":
                True,

            "retrieval_config":
                FROZEN_CONFIG,

            "records":
                NOANSWER_GENERATION_RECORDS,
        },
        f,
        ensure_ascii=False,
        indent=2
    )


print("✅ Saved:")
print(NOANSWER_JSON_PATH)

print(
    "Questions:",
    len(NOANSWER_GENERATION_RECORDS)
)

assert len(
    NOANSWER_GENERATION_RECORDS
) == 4

NameError: name 'NOANSWER_FINAL_RANKINGS' is not defined

In [56]:
# ============================================================
# RECHECK GOLD EVIDENCE COVERAGE
# ============================================================

for config_name, chunks in chunks_by_config.items():

    gold_map = build_gold_chunk_map(
        answerable_questions,
        chunks
    )

    covered = sum(
        bool(gold_map[q["question_id"]])
        for q in answerable_questions
    )

    print(
        f"{config_name:<7}",
        "| gold evidence coverage =",
        f"{covered / len(answerable_questions):.3f}",
        f"({covered}/{len(answerable_questions)})"
    )

    # Show how q057 maps specifically
    q057_gold = gold_map["q057"]

    print(
        f"        q057 relevant chunks:",
        len(q057_gold)
    )

    for chunk_id in sorted(q057_gold):
        print("        -", chunk_id)

ZeroDivisionError: division by zero

## Retrieval experiment comes next

Once the gold file is populated and validated, the next stage will compare:

### Embeddings
- BGE-M3
- multilingual-E5-large
- one newer multilingual embedding candidate

### Retrieval
- BM25
- dense retrieval
- BM25 + dense hybrid fusion

### Metrics
- Recall@1
- Recall@3
- Recall@5
- MRR@10
- nDCG@10
- query latency
- memory / index size

### Then
Only the strongest development configurations will be reranked.

The held-out test split is evaluated once after configuration selection.

In [57]:
# ============================================================
# CELL 23 — GPU BENCHMARK CONFIGURATION
# ============================================================

import torch
import gc
from pathlib import Path

assert torch.cuda.is_available(), (
    "GPU is not enabled. "
    "Kaggle → Settings → Accelerator → GPU"
)

NUM_GPUS = torch.cuda.device_count()

GPU_DEVICES = [
    f"cuda:{i}"
    for i in range(NUM_GPUS)
]

print("GPUs:", GPU_DEVICES)

for i in range(NUM_GPUS):
    print(
        f"{GPU_DEVICES[i]}:",
        torch.cuda.get_device_name(i)
    )

# ------------------------------------------------------------
# GPU optimizations
# ------------------------------------------------------------

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# ------------------------------------------------------------
# Embedding cache
#
# VERY IMPORTANT:
# never recompute corpus embeddings unnecessarily.
# ------------------------------------------------------------

EMBEDDING_CACHE_DIR = (
    WORK_DIR / "embedding_cache"
)

EMBEDDING_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Embedding cache:",
    EMBEDDING_CACHE_DIR
)

GPUs: ['cuda:0', 'cuda:1']
cuda:0: Tesla T4
cuda:1: Tesla T4
Embedding cache: /kaggle/working/docmind_final_v1/embedding_cache


In [93]:
# ============================================================
# CELL 24 — GPU EMBEDDING ENGINE
# ============================================================

import gc
import numpy as np
import torch

from sentence_transformers import SentenceTransformer


EMBEDDING_MODELS = {

    "bge_m3": {
        "model_name": "BAAI/bge-m3",
        "query_prefix": "",
        "document_prefix": "",
    },

    "e5_large": {
        "model_name": "intfloat/multilingual-e5-large",
        "query_prefix": "query: ",
        "document_prefix": "passage: ",
    },

    "qwen3_06b": {
        "model_name": "Qwen/Qwen3-Embedding-0.6B",
        "query_prefix": "",
        "document_prefix": "",
    },
}


def clear_gpu():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def load_embedding_model(model_key):

    cfg = EMBEDDING_MODELS[model_key]

    clear_gpu()

    print(
        f"Loading {cfg['model_name']}..."
    )

    model = SentenceTransformer(
        cfg["model_name"],
        device="cuda:0",
        model_kwargs={
            "torch_dtype": torch.float16
        }
    )

    model.eval()

    print("✅ Loaded:", model_key)

    return model


def encode_documents(
    model,
    model_key,
    texts,
    batch_size=64
):

    cfg = EMBEDDING_MODELS[model_key]

    texts = [
        cfg["document_prefix"] + text
        for text in texts
    ]

    with torch.inference_mode():

        embeddings = model.encode(
            texts,

            batch_size=batch_size,

            show_progress_bar=True,

            convert_to_numpy=True,

            normalize_embeddings=True,

            device="cuda:0",
        )

    return embeddings.astype(
        np.float32
    )


def encode_queries(
    model,
    model_key,
    queries,
    batch_size=64
):

    cfg = EMBEDDING_MODELS[model_key]

    # Qwen3 has a native query prompt.
    if model_key == "qwen3_06b":

        with torch.inference_mode():

            embeddings = model.encode(
                queries,

                prompt_name="query",

                batch_size=batch_size,

                show_progress_bar=True,

                convert_to_numpy=True,

                normalize_embeddings=True,

                device="cuda:0",
            )

    else:

        queries = [
            cfg["query_prefix"] + q
            for q in queries
        ]

        with torch.inference_mode():

            embeddings = model.encode(
                queries,

                batch_size=batch_size,

                show_progress_bar=True,

                convert_to_numpy=True,

                normalize_embeddings=True,

                device="cuda:0",
            )

    return embeddings.astype(
        np.float32
    )

In [94]:
# ============================================================
# CELL 25 — EMBEDDING CACHE
# ============================================================

def safe_name(text):
    return (
        text
        .replace("/", "__")
        .replace("\\", "__")
        .replace(" ", "_")
    )


def get_embedding_paths(
    model_key,
    chunk_config
):

    base = (
        EMBEDDING_CACHE_DIR
        / f"{model_key}__{chunk_config}"
    )

    return {
        "documents": Path(
            str(base) + "__documents.npy"
        ),
        "chunk_ids": Path(
            str(base) + "__chunk_ids.json"
        ),
    }


def get_or_create_document_embeddings(
    model_key,
    chunk_config,
    batch_size=64
):

    chunks = chunks_by_config[
        chunk_config
    ]

    paths = get_embedding_paths(
        model_key,
        chunk_config
    )

    # -----------------------------------
    # CACHE HIT
    # -----------------------------------

    if (
        paths["documents"].exists()
        and
        paths["chunk_ids"].exists()
    ):

        print(
            "♻️ Loading cached embeddings:",
            model_key,
            chunk_config
        )

        embeddings = np.load(
            paths["documents"]
        )

        with open(
            paths["chunk_ids"],
            "r",
            encoding="utf-8"
        ) as f:

            chunk_ids = json.load(f)

        return embeddings, chunk_ids

    # -----------------------------------
    # CACHE MISS
    # -----------------------------------

    print()
    print("=" * 70)
    print(
        "GPU EMBEDDING:",
        model_key,
        "|",
        chunk_config
    )
    print("=" * 70)

    model = load_embedding_model(
        model_key
    )

    texts = [
        c["text"]
        for c in chunks
    ]

    embeddings = encode_documents(
        model,
        model_key,
        texts,
        batch_size=batch_size
    )

    chunk_ids = [
        c["chunk_id"]
        for c in chunks
    ]

    np.save(
        paths["documents"],
        embeddings
    )

    with open(
        paths["chunk_ids"],
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            chunk_ids,
            f,
            ensure_ascii=False
        )

    print(
        "✅ Saved:",
        embeddings.shape
    )

    # Delete model before loading next one
    del model

    clear_gpu()

    return embeddings, chunk_ids

In [95]:
# ============================================================
# CELL 26 — PRECOMPUTE ALL CORPUS EMBEDDINGS
# ============================================================

MODEL_KEYS = [
    "bge_m3",
    "e5_large",
    "qwen3_06b",
]

CHUNK_KEYS = [
    "small",
    "medium",
    "large",
]


for model_key in MODEL_KEYS:

    for chunk_config in CHUNK_KEYS:

        embeddings, chunk_ids = (
            get_or_create_document_embeddings(
                model_key=model_key,
                chunk_config=chunk_config,
                batch_size=64,
            )
        )

        print(
            model_key,
            chunk_config,
            embeddings.shape
        )

        del embeddings
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

♻️ Loading cached embeddings: bge_m3 small
bge_m3 small (1546, 1024)
♻️ Loading cached embeddings: bge_m3 medium
bge_m3 medium (1210, 1024)
♻️ Loading cached embeddings: bge_m3 large
bge_m3 large (1092, 1024)
♻️ Loading cached embeddings: e5_large small
e5_large small (1546, 1024)
♻️ Loading cached embeddings: e5_large medium
e5_large medium (1210, 1024)
♻️ Loading cached embeddings: e5_large large
e5_large large (1092, 1024)
♻️ Loading cached embeddings: qwen3_06b small
qwen3_06b small (1546, 1024)
♻️ Loading cached embeddings: qwen3_06b medium
qwen3_06b medium (1210, 1024)
♻️ Loading cached embeddings: qwen3_06b large
qwen3_06b large (1092, 1024)


In [96]:
# ============================================================
# FIND THE GOLD JSON I UPLOADED TO KAGGLE
# ============================================================

from pathlib import Path
import shutil

matches = list(
    Path("/kaggle").rglob(
        "rag_gold_questions_starter_v1.json"
    )
)

print("Found:", matches)

assert matches, (
    "Upload rag_gold_questions_starter_v1.json "
    "to Kaggle first."
)

SOURCE_GOLD = matches[0]

GOLD_PATH = (
    WORK_DIR /
    "rag_gold_questions.json"
)

shutil.copy(
    SOURCE_GOLD,
    GOLD_PATH
)

print("✅ Copied benchmark")
print("FROM:", SOURCE_GOLD)
print("TO  :", GOLD_PATH)

Found: [PosixPath('/kaggle/input/datasets/ripperdzz/gold-qst/rag_gold_questions_starter_v1.json')]
✅ Copied benchmark
FROM: /kaggle/input/datasets/ripperdzz/gold-qst/rag_gold_questions_starter_v1.json
TO  : /kaggle/working/docmind_research/rag_gold_questions.json


In [97]:
# ============================================================
# CELL 27 — RETRIEVAL METRICS
# ============================================================

import math
import numpy as np
import pandas as pd


def recall_at_k(ranking, gold_ids, k):
    gold = set(gold_ids)

    if not gold:
        return 0.0

    retrieved = set(ranking[:k])

    return len(gold & retrieved) / len(gold)


def mrr_at_k(ranking, gold_ids, k=10):
    gold = set(gold_ids)

    for rank, chunk_id in enumerate(ranking[:k], start=1):
        if chunk_id in gold:
            return 1.0 / rank

    return 0.0


def ndcg_at_k(ranking, gold_ids, k=10):
    gold = set(gold_ids)

    if not gold:
        return 0.0

    dcg = 0.0

    for rank, chunk_id in enumerate(ranking[:k], start=1):
        if chunk_id in gold:
            dcg += 1.0 / math.log2(rank + 1)

    ideal_hits = min(len(gold), k)

    idcg = sum(
        1.0 / math.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    return dcg / idcg if idcg else 0.0


def evaluate_rankings(
    questions,
    rankings,
    gold_map
):

    rows = []

    for q in questions:

        qid = q["question_id"]

        ranking = rankings[qid]
        gold = gold_map[qid]

        rows.append({
            "question_id": qid,
            "language": q["language"],
            "question_type": q["question_type"],

            "Recall@1": recall_at_k(
                ranking, gold, 1
            ),

            "Recall@3": recall_at_k(
                ranking, gold, 3
            ),

            "Recall@5": recall_at_k(
                ranking, gold, 5
            ),

            "MRR@10": mrr_at_k(
                ranking, gold, 10
            ),

            "nDCG@10": ndcg_at_k(
                ranking, gold, 10
            ),
        })

    df = pd.DataFrame(rows)

    return df, {
        "Recall@1": df["Recall@1"].mean(),
        "Recall@3": df["Recall@3"].mean(),
        "Recall@5": df["Recall@5"].mean(),
        "MRR@10": df["MRR@10"].mean(),
        "nDCG@10": df["nDCG@10"].mean(),
    }


print("✅ Metrics ready")

✅ Metrics ready


In [98]:
# ============================================================
# CELL 28 — RETRIEVAL ENGINES
# ============================================================

import re
import faiss
from rank_bm25 import BM25Okapi


def bm25_tokenize(text):
    return re.findall(
        r"\w+",
        text.lower(),
        flags=re.UNICODE
    )


def build_bm25(chunks):

    tokens = [
        bm25_tokenize(c["text"])
        for c in chunks
    ]

    return BM25Okapi(tokens)


def bm25_rank(
    bm25,
    chunks,
    question,
    top_k=50
):

    scores = bm25.get_scores(
        bm25_tokenize(question)
    )

    indices = np.argsort(scores)[::-1][:top_k]

    return [
        chunks[i]["chunk_id"]
        for i in indices
    ]


def build_faiss_index(embeddings):

    embeddings = np.asarray(
        embeddings,
        dtype=np.float32
    )

    index = faiss.IndexFlatIP(
        embeddings.shape[1]
    )

    index.add(embeddings)

    return index


def dense_rank_all(
    index,
    query_embeddings,
    chunks,
    questions,
    top_k=50
):

    scores, indices = index.search(
        np.asarray(
            query_embeddings,
            dtype=np.float32
        ),
        top_k
    )

    rankings = {}

    for q, row in zip(
        questions,
        indices
    ):

        rankings[q["question_id"]] = [
            chunks[i]["chunk_id"]
            for i in row
            if i >= 0
        ]

    return rankings


def rrf(
    dense_ranking,
    bm25_ranking,
    rrf_k=60,
    top_k=50
):

    scores = {}

    for ranking in [
        dense_ranking,
        bm25_ranking
    ]:

        for rank, chunk_id in enumerate(
            ranking,
            start=1
        ):

            scores[chunk_id] = (
                scores.get(chunk_id, 0.0)
                +
                1.0 / (rrf_k + rank)
            )

    ordered = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        chunk_id
        for chunk_id, _ in ordered[:top_k]
    ]


print("✅ BM25 / Dense / Hybrid ready")

✅ BM25 / Dense / Hybrid ready


In [99]:
# ============================================================
# CELL 29 — CACHE DEV QUERY EMBEDDINGS
# ============================================================

QUERY_CACHE_DIR = (
    WORK_DIR / "query_embedding_cache"
)

QUERY_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def get_dev_query_embeddings(
    model_key,
    batch_size=64
):

    cache_path = (
        QUERY_CACHE_DIR /
        f"{model_key}__dev.npy"
    )

    if cache_path.exists():

        print(
            f"♻️ {model_key}: cached"
        )

        return np.load(cache_path)

    print(
        f"🔥 Embedding DEV queries: {model_key}"
    )

    model = load_embedding_model(
        model_key
    )

    queries = [
        q["question"]
        for q in dev_questions
    ]

    embeddings = encode_queries(
        model,
        model_key,
        queries,
        batch_size=batch_size
    )

    np.save(
        cache_path,
        embeddings
    )

    del model
    clear_gpu()

    return embeddings


DEV_QUERY_EMBEDDINGS = {}

for model_key in MODEL_KEYS:

    DEV_QUERY_EMBEDDINGS[model_key] = (
        get_dev_query_embeddings(
            model_key
        )
    )

    print(
        model_key,
        DEV_QUERY_EMBEDDINGS[
            model_key
        ].shape
    )

♻️ bge_m3: cached
bge_m3 (0,)
♻️ e5_large: cached
e5_large (0,)
♻️ qwen3_06b: cached
qwen3_06b (0,)


In [100]:
# ============================================================
# CELL 30 — FULL DEVELOPMENT RETRIEVAL BENCHMARK
# ============================================================

import time

DEV_RESULTS = []
DEV_DETAILS = {}
DEV_RANKINGS = {}


for chunk_config in CHUNK_KEYS:

    print("\n" + "=" * 90)
    print("CHUNK CONFIG:", chunk_config)
    print("=" * 90)

    chunks = chunks_by_config[
        chunk_config
    ]

    gold_map = build_gold_chunk_map(
        dev_questions,
        chunks
    )

    # Safety check
    missing = [
        q["question_id"]
        for q in dev_questions
        if not gold_map[q["question_id"]]
    ]

    assert not missing, (
        f"Missing gold for {chunk_config}: "
        f"{missing}"
    )

    # ========================================================
    # BM25
    # ========================================================

    bm25 = build_bm25(chunks)

    bm25_rankings = {}

    start = time.perf_counter()

    for q in dev_questions:

        bm25_rankings[
            q["question_id"]
        ] = bm25_rank(
            bm25,
            chunks,
            q["question"],
            top_k=50
        )

    bm25_latency = (
        time.perf_counter() - start
    ) / len(dev_questions)

    bm25_df, bm25_summary = (
        evaluate_rankings(
            dev_questions,
            bm25_rankings,
            gold_map
        )
    )

    DEV_RESULTS.append({
        "chunk_config":
            chunk_config,

        "embedding_model":
            "none",

        "retriever":
            "BM25",

        "latency_ms":
            bm25_latency * 1000,

        **bm25_summary
    })

    DEV_DETAILS[
        (
            chunk_config,
            "none",
            "BM25"
        )
    ] = bm25_df

    DEV_RANKINGS[
        (
            chunk_config,
            "none",
            "BM25"
        )
    ] = bm25_rankings

    # ========================================================
    # EMBEDDING MODELS
    # ========================================================

    for model_key in MODEL_KEYS:

        print(
            f"→ {model_key}"
        )

        embedding_paths = (
            get_embedding_paths(
                model_key,
                chunk_config
            )
        )

        document_embeddings = np.load(
            embedding_paths[
                "documents"
            ]
        )

        query_embeddings = (
            DEV_QUERY_EMBEDDINGS[
                model_key
            ]
        )

        index = build_faiss_index(
            document_embeddings
        )

        # ====================================================
        # DENSE
        # ====================================================

        start = time.perf_counter()

        dense_rankings = dense_rank_all(
            index,
            query_embeddings,
            chunks,
            dev_questions,
            top_k=50
        )

        dense_latency = (
            time.perf_counter() - start
        ) / len(dev_questions)

        dense_df, dense_summary = (
            evaluate_rankings(
                dev_questions,
                dense_rankings,
                gold_map
            )
        )

        DEV_RESULTS.append({
            "chunk_config":
                chunk_config,

            "embedding_model":
                model_key,

            "retriever":
                "dense",

            "latency_ms":
                dense_latency * 1000,

            **dense_summary
        })

        DEV_DETAILS[
            (
                chunk_config,
                model_key,
                "dense"
            )
        ] = dense_df

        DEV_RANKINGS[
            (
                chunk_config,
                model_key,
                "dense"
            )
        ] = dense_rankings

        # ====================================================
        # HYBRID RRF
        # ====================================================

        start = time.perf_counter()

        hybrid_rankings = {}

        for q in dev_questions:

            qid = q["question_id"]

            hybrid_rankings[qid] = rrf(
                dense_rankings[qid],
                bm25_rankings[qid],
                rrf_k=60,
                top_k=50
            )

        hybrid_latency = (
            time.perf_counter() - start
        ) / len(dev_questions)

        hybrid_df, hybrid_summary = (
            evaluate_rankings(
                dev_questions,
                hybrid_rankings,
                gold_map
            )
        )

        DEV_RESULTS.append({
            "chunk_config":
                chunk_config,

            "embedding_model":
                model_key,

            "retriever":
                "hybrid",

            # Includes only fusion overhead,
            # not embedding cost.
            "latency_ms":
                (
                    dense_latency
                    +
                    bm25_latency
                    +
                    hybrid_latency
                ) * 1000,

            **hybrid_summary
        })

        DEV_DETAILS[
            (
                chunk_config,
                model_key,
                "hybrid"
            )
        ] = hybrid_df

        DEV_RANKINGS[
            (
                chunk_config,
                model_key,
                "hybrid"
            )
        ] = hybrid_rankings

        del index
        del document_embeddings


DEV_SUMMARY = pd.DataFrame(
    DEV_RESULTS
)

print("\n✅ DEV benchmark finished")


CHUNK CONFIG: small
→ bge_m3


ValueError: not enough values to unpack (expected 2, got 1)

In [ ]:
# ============================================================
# CELL 31 — DEV LEADERBOARD
# ============================================================

DEV_SUMMARY = (
    DEV_SUMMARY
    .sort_values(
        by=[
            "Recall@5",
            "MRR@10",
            "nDCG@10"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    DEV_SUMMARY.style.format({
        "Recall@1": "{:.4f}",
        "Recall@3": "{:.4f}",
        "Recall@5": "{:.4f}",
        "MRR@10": "{:.4f}",
        "nDCG@10": "{:.4f}",
        "latency_ms": "{:.3f}",
    })
)

In [ ]:
# ============================================================
# CELL 32 — SAVE DEV RESULTS
# ============================================================

DEV_SUMMARY.to_csv(
    WORK_DIR /
    "dev_retrieval_summary.csv",
    index=False
)


for key, df in DEV_DETAILS.items():

    chunk_config, model, retriever = key

    filename = (
        f"dev__"
        f"{chunk_config}__"
        f"{model}__"
        f"{retriever}.csv"
    )

    df.to_csv(
        WORK_DIR / filename,
        index=False
    )


print(
    "✅ Results saved to:",
    WORK_DIR
)

In [ ]:
# ============================================================
# CELL 33 — BENCHMARK PLOTTING SETUP
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PLOT_DIR = WORK_DIR / "benchmark_plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

plot_df = DEV_SUMMARY.copy()

plot_df["label"] = (
    plot_df["embedding_model"].replace({
        "qwen3_06b": "Qwen3-0.6B",
        "e5_large": "E5-large",
        "bge_m3": "BGE-M3",
        "none": "BM25"
    })
    + " | "
    + plot_df["retriever"]
    + " | "
    + plot_df["chunk_config"]
)

print("Configurations:", len(plot_df))

In [ ]:
# ============================================================
# CELL 34 — RECALL@5 LEADERBOARD
# ============================================================

top = (
    plot_df
    .sort_values("Recall@5", ascending=True)
)

fig, ax = plt.subplots(figsize=(11, 9))

bars = ax.barh(
    top["label"],
    top["Recall@5"]
)

ax.set_xlabel("Recall@5")
ax.set_ylabel("")
ax.set_title(
    "DocMind DEV Retrieval Benchmark — Recall@5"
)

ax.set_xlim(0.75, 1.02)

ax.grid(
    axis="x",
    alpha=0.25
)

for bar, value in zip(
    bars,
    top["Recall@5"]
):
    ax.text(
        value + 0.003,
        bar.get_y() + bar.get_height()/2,
        f"{value:.3f}",
        va="center",
        fontsize=9
    )

plt.tight_layout()

path = PLOT_DIR / "01_recall5_leaderboard.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", path)

In [ ]:
# ============================================================
# CELL 35 — TOP-K RETRIEVAL CURVES
# ============================================================

top_n = 10

best = (
    plot_df
    .sort_values(
        ["Recall@5", "MRR@10"],
        ascending=False
    )
    .head(top_n)
    .copy()
)

metrics = [
    "Recall@1",
    "Recall@3",
    "Recall@5"
]

x = np.arange(len(best))
width = 0.25

fig, ax = plt.subplots(
    figsize=(14, 7)
)

for i, metric in enumerate(metrics):

    ax.bar(
        x + (i - 1) * width,
        best[metric],
        width,
        label=metric
    )

ax.set_xticks(x)

ax.set_xticklabels(
    best["label"],
    rotation=55,
    ha="right"
)

ax.set_ylim(0.6, 1.03)

ax.set_ylabel("Recall")
ax.set_title(
    "Top Retrieval Configurations — Recall Growth with K"
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

path = PLOT_DIR / "02_recall_at_k.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", path)

In [ ]:
# ============================================================
# CELL 36 — DENSE RETRIEVAL HEATMAP
# ============================================================

dense = plot_df[
    plot_df["retriever"] == "dense"
].copy()

model_order = [
    "qwen3_06b",
    "e5_large",
    "bge_m3"
]

chunk_order = [
    "small",
    "medium",
    "large"
]

matrix = (
    dense
    .pivot(
        index="embedding_model",
        columns="chunk_config",
        values="Recall@5"
    )
    .reindex(
        index=model_order,
        columns=chunk_order
    )
)

fig, ax = plt.subplots(
    figsize=(8, 5)
)

im = ax.imshow(
    matrix.values,
    aspect="auto"
)

ax.set_xticks(
    np.arange(len(chunk_order))
)

ax.set_xticklabels(
    ["Small", "Medium", "Large"]
)

ax.set_yticks(
    np.arange(len(model_order))
)

ax.set_yticklabels(
    [
        "Qwen3-Embedding-0.6B",
        "multilingual-E5-large",
        "BGE-M3"
    ]
)

ax.set_title(
    "Dense Retrieval Recall@5\nEmbedding Model × Chunk Size"
)

for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):

        value = matrix.iloc[i, j]

        ax.text(
            j,
            i,
            f"{value:.3f}",
            ha="center",
            va="center"
        )

fig.colorbar(
    im,
    ax=ax,
    label="Recall@5"
)

plt.tight_layout()

path = PLOT_DIR / "03_dense_heatmap.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", path)

In [ ]:
# ============================================================
# CELL 37 — DOES HYBRID ACTUALLY HELP?
# ============================================================

models_only = plot_df[
    plot_df["embedding_model"] != "none"
].copy()

comparison = (
    models_only
    .pivot_table(
        index=[
            "chunk_config",
            "embedding_model"
        ],
        columns="retriever",
        values=[
            "Recall@1",
            "Recall@5"
        ]
    )
)

comparison[
    ("delta", "Recall@1")
] = (
    comparison[
        ("Recall@1", "hybrid")
    ]
    -
    comparison[
        ("Recall@1", "dense")
    ]
)

comparison[
    ("delta", "Recall@5")
] = (
    comparison[
        ("Recall@5", "hybrid")
    ]
    -
    comparison[
        ("Recall@5", "dense")
    ]
)

delta = comparison["delta"].reset_index()

delta["label"] = (
    delta["embedding_model"].replace({
        "qwen3_06b": "Qwen3",
        "e5_large": "E5",
        "bge_m3": "BGE-M3"
    })
    + " | "
    + delta["chunk_config"]
)

x = np.arange(len(delta))
width = 0.35

fig, ax = plt.subplots(
    figsize=(12, 6)
)

ax.bar(
    x - width/2,
    delta["Recall@1"],
    width,
    label="Δ Recall@1"
)

ax.bar(
    x + width/2,
    delta["Recall@5"],
    width,
    label="Δ Recall@5"
)

ax.axhline(
    0,
    linewidth=1
)

ax.set_xticks(x)

ax.set_xticklabels(
    delta["label"],
    rotation=45,
    ha="right"
)

ax.set_ylabel(
    "Hybrid − Dense"
)

ax.set_title(
    "Impact of Adding BM25 Hybrid Fusion"
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

path = PLOT_DIR / "04_hybrid_delta.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", path)

In [ ]:
# ============================================================
# CELL 38 — QUALITY TRADE-OFF
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 7)
)

for _, row in plot_df.iterrows():

    ax.scatter(
        row["Recall@5"],
        row["MRR@10"],
        s=80
    )

    ax.annotate(
        row["label"],
        (
            row["Recall@5"],
            row["MRR@10"]
        ),
        xytext=(5, 4),
        textcoords="offset points",
        fontsize=7
    )

ax.set_xlabel("Recall@5")
ax.set_ylabel("MRR@10")

ax.set_title(
    "Retrieval Quality Trade-off: Coverage vs Ranking"
)

ax.grid(alpha=0.25)

plt.tight_layout()

path = PLOT_DIR / "05_recall5_vs_mrr.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", path)

In [ ]:
# ============================================================
# CELL 40 — PERFORMANCE BY LANGUAGE
# ============================================================

language_rows = []

for key, detail_df in DEV_DETAILS.items():

    chunk_config, model, retriever = key

    grouped = (
        detail_df
        .groupby("language")[
            [
                "Recall@1",
                "Recall@3",
                "Recall@5",
                "MRR@10",
                "nDCG@10"
            ]
        ]
        .mean()
        .reset_index()
    )

    for _, row in grouped.iterrows():

        language_rows.append({
            "chunk_config": chunk_config,
            "embedding_model": model,
            "retriever": retriever,
            "language": row["language"],
            "Recall@5": row["Recall@5"],
            "MRR@10": row["MRR@10"]
        })


LANGUAGE_RESULTS = pd.DataFrame(
    language_rows
)

best_configs = (
    DEV_SUMMARY
    .head(6)[
        [
            "chunk_config",
            "embedding_model",
            "retriever"
        ]
    ]
)

selected = LANGUAGE_RESULTS.merge(
    best_configs,
    on=[
        "chunk_config",
        "embedding_model",
        "retriever"
    ]
)

selected["label"] = (
    selected["embedding_model"]
    .replace({
        "qwen3_06b": "Qwen3",
        "e5_large": "E5",
        "bge_m3": "BGE-M3",
        "none": "BM25"
    })
    + " | "
    + selected["retriever"]
    + " | "
    + selected["chunk_config"]
)


pivot = selected.pivot(
    index="label",
    columns="language",
    values="Recall@5"
)

pivot.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.ylabel("Recall@5")
plt.xlabel("")

plt.title(
    "DEV Recall@5 by Query Language"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.ylim(0, 1.05)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

path = PLOT_DIR / "07_language_breakdown.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", path)

display(pivot)

In [ ]:
# ============================================================
# CELL 41 — PERFORMANCE BY QUESTION TYPE
# ============================================================

type_rows = []

for key, detail_df in DEV_DETAILS.items():

    chunk_config, model, retriever = key

    grouped = (
        detail_df
        .groupby("question_type")[
            [
                "Recall@1",
                "Recall@5",
                "MRR@10"
            ]
        ]
        .mean()
        .reset_index()
    )

    for _, row in grouped.iterrows():

        type_rows.append({
            "chunk_config": chunk_config,
            "embedding_model": model,
            "retriever": retriever,
            "question_type": row["question_type"],
            "Recall@5": row["Recall@5"],
            "MRR@10": row["MRR@10"]
        })


TYPE_RESULTS = pd.DataFrame(type_rows)


# Compare top 5 configurations only
top5 = DEV_SUMMARY.head(5)[
    [
        "chunk_config",
        "embedding_model",
        "retriever"
    ]
]

selected = TYPE_RESULTS.merge(
    top5,
    on=[
        "chunk_config",
        "embedding_model",
        "retriever"
    ]
)

selected["label"] = (
    selected["embedding_model"]
    .replace({
        "qwen3_06b": "Qwen3",
        "e5_large": "E5",
        "bge_m3": "BGE-M3",
        "none": "BM25"
    })
    + " | "
    + selected["retriever"]
    + " | "
    + selected["chunk_config"]
)


pivot = selected.pivot(
    index="question_type",
    columns="label",
    values="Recall@5"
)

ax = pivot.plot(
    kind="bar",
    figsize=(14, 7)
)

ax.set_ylabel("Recall@5")
ax.set_xlabel("Question type")

ax.set_title(
    "Retrieval Performance by Question Type"
)

ax.set_ylim(0, 1.05)

ax.grid(
    axis="y",
    alpha=0.25
)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

path = PLOT_DIR / "08_question_type_breakdown.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", path)

display(pivot)

In [ ]:
# ============================================================
# CELL 42 — REPORT TABLE
# ============================================================

REPORT_TABLE = (
    DEV_SUMMARY[
        [
            "chunk_config",
            "embedding_model",
            "retriever",
            "Recall@1",
            "Recall@3",
            "Recall@5",
            "MRR@10",
            "nDCG@10",
            "latency_ms"
        ]
    ]
    .copy()
)

REPORT_TABLE["embedding_model"] = (
    REPORT_TABLE["embedding_model"]
    .replace({
        "qwen3_06b":
            "Qwen3-Embedding-0.6B",

        "e5_large":
            "multilingual-E5-large",

        "bge_m3":
            "BGE-M3",

        "none":
            "-"
    })
)

display(
    REPORT_TABLE.style
    .format({
        "Recall@1": "{:.3f}",
        "Recall@3": "{:.3f}",
        "Recall@5": "{:.3f}",
        "MRR@10": "{:.3f}",
        "nDCG@10": "{:.3f}",
        "latency_ms": "{:.3f}",
    })
    .background_gradient(
        subset=[
            "Recall@1",
            "Recall@3",
            "Recall@5",
            "MRR@10",
            "nDCG@10"
        ]
    )
)

In [ ]:
# ============================================================
# CELL 43 — EVIDENCE-GROUP GOLD LABELS
# ============================================================

def build_gold_evidence_groups(
    questions,
    chunks
):

    result = {}

    for q in questions:

        groups = []

        for evidence in q.get(
            "gold_evidence",
            []
        ):

            matching_chunks = {
                chunk["chunk_id"]
                for chunk in chunks
                if evidence_matches_chunk(
                    evidence,
                    chunk
                )
            }

            groups.append(
                matching_chunks
            )

        result[
            q["question_id"]
        ] = groups

    return result

In [ ]:
def evidence_recall_at_k(
    ranking,
    evidence_groups,
    k
):

    if not evidence_groups:
        return 0.0

    retrieved = set(
        ranking[:k]
    )

    covered = sum(
        bool(
            retrieved & group
        )
        for group in evidence_groups
    )

    return covered / len(
        evidence_groups
    )

In [ ]:
def complete_evidence_at_k(
    ranking,
    evidence_groups,
    k
):

    if not evidence_groups:
        return 0.0

    retrieved = set(
        ranking[:k]
    )

    return float(
        all(
            bool(retrieved & group)
            for group in evidence_groups
        )
    )

In [ ]:
# ============================================================
# CHECK CompleteEvidence@5 ON ALL DEV CONFIGURATIONS
# ============================================================

COMPLETE_RESULTS = []

for key, rankings in DEV_RANKINGS.items():

    chunk_config, model, retriever = key

    chunks = chunks_by_config[chunk_config]

    evidence_groups = build_gold_evidence_groups(
        dev_questions,
        chunks
    )

    per_question = []

    for q in dev_questions:

        qid = q["question_id"]

        score = complete_evidence_at_k(
            rankings[qid],
            evidence_groups[qid],
            k=5
        )

        per_question.append(score)

    COMPLETE_RESULTS.append({
        "chunk_config": chunk_config,
        "embedding_model": model,
        "retriever": retriever,
        "CompleteEvidence@5": np.mean(per_question),
        "complete_questions": int(np.sum(per_question)),
        "total_questions": len(per_question),
    })


COMPLETE_DF = (
    pd.DataFrame(COMPLETE_RESULTS)
    .sort_values(
        "CompleteEvidence@5",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    COMPLETE_DF.style.format({
        "CompleteEvidence@5": "{:.4f}"
    })
)

In [ ]:
DEV_SUMMARY_V2 = DEV_SUMMARY.merge(
    COMPLETE_DF[
        [
            "chunk_config",
            "embedding_model",
            "retriever",
            "CompleteEvidence@5"
        ]
    ],
    on=[
        "chunk_config",
        "embedding_model",
        "retriever"
    ],
    how="left"
)

DEV_SUMMARY_V2 = (
    DEV_SUMMARY_V2
    .sort_values(
        [
            "CompleteEvidence@5",
            "Recall@5",
            "MRR@10"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    DEV_SUMMARY_V2.style.format({
        "Recall@1": "{:.4f}",
        "Recall@3": "{:.4f}",
        "Recall@5": "{:.4f}",
        "CompleteEvidence@5": "{:.4f}",
        "MRR@10": "{:.4f}",
        "nDCG@10": "{:.4f}",
    })
)

In [ ]:
# ============================================================
# CORRECT EVIDENCE-LEVEL RETRIEVAL METRICS
# ============================================================

def evidence_recall_at_k(
    ranking,
    evidence_groups,
    k
):
    if not evidence_groups:
        return 0.0

    retrieved = set(ranking[:k])

    covered = sum(
        bool(retrieved & group)
        for group in evidence_groups
    )

    return covered / len(evidence_groups)


EVIDENCE_RESULTS = []

for key, rankings in DEV_RANKINGS.items():

    chunk_config, model, retriever = key

    groups_map = build_gold_evidence_groups(
        dev_questions,
        chunks_by_config[chunk_config]
    )

    rows = []

    for q in dev_questions:

        qid = q["question_id"]
        ranking = rankings[qid]
        groups = groups_map[qid]

        rows.append({
            "EvidenceRecall@1":
                evidence_recall_at_k(
                    ranking, groups, 1
                ),

            "EvidenceRecall@3":
                evidence_recall_at_k(
                    ranking, groups, 3
                ),

            "EvidenceRecall@5":
                evidence_recall_at_k(
                    ranking, groups, 5
                ),

            "CompleteEvidence@5":
                complete_evidence_at_k(
                    ranking, groups, 5
                )
        })

    df = pd.DataFrame(rows)

    EVIDENCE_RESULTS.append({
        "chunk_config": chunk_config,
        "embedding_model": model,
        "retriever": retriever,

        "EvidenceRecall@1":
            df["EvidenceRecall@1"].mean(),

        "EvidenceRecall@3":
            df["EvidenceRecall@3"].mean(),

        "EvidenceRecall@5":
            df["EvidenceRecall@5"].mean(),

        "CompleteEvidence@5":
            df["CompleteEvidence@5"].mean(),
    })


EVIDENCE_SUMMARY = (
    pd.DataFrame(EVIDENCE_RESULTS)
    .sort_values(
        [
            "CompleteEvidence@5",
            "EvidenceRecall@5",
            "EvidenceRecall@3"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    EVIDENCE_SUMMARY.style.format({
        "EvidenceRecall@1": "{:.4f}",
        "EvidenceRecall@3": "{:.4f}",
        "EvidenceRecall@5": "{:.4f}",
        "CompleteEvidence@5": "{:.4f}",
    })
)

In [ ]:
!pip install -q FlagEmbedding

In [ ]:
from FlagEmbedding import FlagReranker
import torch
import gc

reranker = FlagReranker(
    "BAAI/bge-reranker-v2-m3",
    use_fp16=True
)

print("✅ Reranker loaded")

In [ ]:
# ============================================================
# RERANK TOP-N RETRIEVAL RESULTS
# ============================================================

def rerank_results(
    questions,
    rankings,
    chunks,
    top_n=30,
    final_k=50
):

    chunk_lookup = {
        c["chunk_id"]: c
        for c in chunks
    }

    reranked = {}

    for q in tqdm(
        questions,
        desc="Reranking"
    ):

        qid = q["question_id"]

        candidate_ids = rankings[qid][:top_n]

        pairs = [
            [
                q["question"],
                chunk_lookup[cid]["text"]
            ]
            for cid in candidate_ids
        ]

        scores = reranker.compute_score(
            pairs,
            normalize=True
        )

        ranked_pairs = sorted(
            zip(candidate_ids, scores),
            key=lambda x: x[1],
            reverse=True
        )

        reranked_ids = [
            cid
            for cid, score in ranked_pairs
        ]

        # Keep untouched results after reranked candidates
        remainder = [
            cid
            for cid in rankings[qid]
            if cid not in set(reranked_ids)
        ]

        reranked[qid] = (
            reranked_ids + remainder
        )[:final_k]

    return reranked

In [ ]:
# ============================================================
# RERANKER — DIRECT TRANSFORMERS VERSION
# ============================================================

import gc
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

RERANKER_NAME = "BAAI/bge-reranker-v2-m3"
RERANK_DEVICE = torch.device("cuda:0")

# Clean up previous models
gc.collect()
torch.cuda.empty_cache()

print("GPU:", torch.cuda.get_device_name(0))
print("Loading:", RERANKER_NAME)

rerank_tokenizer = AutoTokenizer.from_pretrained(
    RERANKER_NAME
)

rerank_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        RERANKER_NAME,
        torch_dtype=torch.float16
    )
    .to(RERANK_DEVICE)
)

rerank_model.eval()

print("✅ Reranker loaded on cuda:0")

In [ ]:
# ============================================================
# RERANKER SMOKE TEST
# ============================================================

test_pairs = [
    (
        "What is machine learning?",
        "Machine learning allows computers to learn from data."
    ),
    (
        "What is machine learning?",
        "Paris is the capital of France."
    ),
]

with torch.inference_mode():

    inputs = rerank_tokenizer(
        [x[0] for x in test_pairs],
        [x[1] for x in test_pairs],
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(RERANK_DEVICE)
        for k, v in inputs.items()
    }

    scores = (
        rerank_model(**inputs)
        .logits
        .view(-1)
        .float()
        .cpu()
        .numpy()
    )

print(scores)

In [ ]:
# ============================================================
# FAST BATCHED RERANKING
# ============================================================

from tqdm.auto import tqdm
import numpy as np


def score_reranker_pairs(
    queries,
    passages,
    batch_size=16,
    max_length=1024
):

    assert len(queries) == len(passages)

    all_scores = []

    for start in tqdm(
        range(0, len(queries), batch_size),
        desc="Reranker batches"
    ):

        end = min(
            start + batch_size,
            len(queries)
        )

        batch_queries = queries[start:end]
        batch_passages = passages[start:end]

        encoded = rerank_tokenizer(
            batch_queries,
            batch_passages,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(
                RERANK_DEVICE,
                non_blocking=True
            )
            for key, value in encoded.items()
        }

        with torch.inference_mode():

            logits = (
                rerank_model(**encoded)
                .logits
                .view(-1)
            )

        all_scores.extend(
            logits
            .float()
            .cpu()
            .numpy()
            .tolist()
        )

    return np.asarray(
        all_scores,
        dtype=np.float32
    )


def rerank_results_batched(
    questions,
    rankings,
    chunks,
    top_n=30,
    final_k=50,
    batch_size=16
):

    chunk_lookup = {
        c["chunk_id"]: c
        for c in chunks
    }

    # --------------------------------------------------------
    # Flatten every query/candidate pair
    # --------------------------------------------------------

    records = []

    for q in questions:

        qid = q["question_id"]

        candidate_ids = rankings[qid][:top_n]

        for cid in candidate_ids:

            records.append({
                "question_id": qid,
                "chunk_id": cid,
                "query": q["question"],
                "passage": chunk_lookup[cid]["text"]
            })

    print(
        "Pairs to rerank:",
        len(records)
    )

    queries = [
        r["query"]
        for r in records
    ]

    passages = [
        r["passage"]
        for r in records
    ]

    # --------------------------------------------------------
    # GPU scoring
    # --------------------------------------------------------

    scores = score_reranker_pairs(
        queries,
        passages,
        batch_size=batch_size,
        max_length=1024
    )

    for record, score in zip(
        records,
        scores
    ):
        record["score"] = float(score)

    # --------------------------------------------------------
    # Reconstruct rankings
    # --------------------------------------------------------

    reranked = {}

    for q in questions:

        qid = q["question_id"]

        candidate_records = [
            r
            for r in records
            if r["question_id"] == qid
        ]

        candidate_records.sort(
            key=lambda x: x["score"],
            reverse=True
        )

        reranked_ids = [
            r["chunk_id"]
            for r in candidate_records
        ]

        candidate_set = set(
            reranked_ids
        )

        remainder = [
            cid
            for cid in rankings[qid]
            if cid not in candidate_set
        ]

        reranked[qid] = (
            reranked_ids
            +
            remainder
        )[:final_k]

    return reranked

In [ ]:
# ============================================================
# TEST RERANKING ON CURRENT DEV WINNER
# ============================================================

key = (
    "medium",
    "qwen3_06b",
    "dense"
)

print("Reranking:", key)

MEDIUM_QWEN_RERANKED = rerank_results_batched(
    questions=dev_questions,

    rankings=DEV_RANKINGS[key],

    chunks=chunks_by_config[
        "medium"
    ],

    top_n=30,

    batch_size=16
)

print("✅ Finished")

In [ ]:
# ============================================================
# BEFORE vs AFTER — CURRENT WINNER
# ============================================================

key = (
    "medium",
    "qwen3_06b",
    "dense"
)

groups_map = build_gold_evidence_groups(
    dev_questions,
    chunks_by_config["medium"]
)


def summarize_evidence_metrics(
    rankings,
    groups_map
):

    rows = []

    for q in dev_questions:

        qid = q["question_id"]
        groups = groups_map[qid]

        rows.append({

            "EvidenceRecall@1":
                evidence_recall_at_k(
                    rankings[qid],
                    groups,
                    1
                ),

            "EvidenceRecall@3":
                evidence_recall_at_k(
                    rankings[qid],
                    groups,
                    3
                ),

            "EvidenceRecall@5":
                evidence_recall_at_k(
                    rankings[qid],
                    groups,
                    5
                ),

            "CompleteEvidence@5":
                complete_evidence_at_k(
                    rankings[qid],
                    groups,
                    5
                ),
        })

    df = pd.DataFrame(rows)

    return df.mean()


before = summarize_evidence_metrics(
    DEV_RANKINGS[key],
    groups_map
)

after = summarize_evidence_metrics(
    MEDIUM_QWEN_RERANKED,
    groups_map
)


comparison = pd.DataFrame({
    "Before reranker": before,
    "After reranker": after
})

display(
    comparison.style.format(
        "{:.4f}"
    )
)

In [ ]:
FINALISTS = [
    ("medium", "qwen3_06b", "dense"),
    ("large",  "qwen3_06b", "dense"),
    ("small",  "qwen3_06b", "dense"),
    ("medium", "qwen3_06b", "hybrid"),
]


RERANKED_DEV = {}


for key in FINALISTS:

    chunk_config, model, retriever = key

    print("\n" + "=" * 80)
    print("RERANK:", key)
    print("=" * 80)

    RERANKED_DEV[key] = (
        rerank_results_batched(

            questions=dev_questions,

            rankings=DEV_RANKINGS[key],

            chunks=chunks_by_config[
                chunk_config
            ],

            top_n=30,

            batch_size=16
        )
    )


print("\n✅ All finalists reranked")

In [ ]:
# ============================================================
# EVALUATE ALL RERANKED FINALISTS
# ============================================================

RERANK_RESULTS = []

for key, rankings in RERANKED_DEV.items():

    chunk_config, model, retriever = key

    groups_map = build_gold_evidence_groups(
        dev_questions,
        chunks_by_config[chunk_config]
    )

    rows = []

    for q in dev_questions:

        qid = q["question_id"]
        groups = groups_map[qid]

        rows.append({
            "EvidenceRecall@1":
                evidence_recall_at_k(
                    rankings[qid],
                    groups,
                    1
                ),

            "EvidenceRecall@3":
                evidence_recall_at_k(
                    rankings[qid],
                    groups,
                    3
                ),

            "EvidenceRecall@5":
                evidence_recall_at_k(
                    rankings[qid],
                    groups,
                    5
                ),

            "CompleteEvidence@5":
                complete_evidence_at_k(
                    rankings[qid],
                    groups,
                    5
                ),
        })

    df = pd.DataFrame(rows)

    RERANK_RESULTS.append({
        "chunk_config": chunk_config,
        "embedding_model": model,
        "base_retriever": retriever,

        "EvidenceRecall@1":
            df["EvidenceRecall@1"].mean(),

        "EvidenceRecall@3":
            df["EvidenceRecall@3"].mean(),

        "EvidenceRecall@5":
            df["EvidenceRecall@5"].mean(),

        "CompleteEvidence@5":
            df["CompleteEvidence@5"].mean(),
    })


RERANK_SUMMARY = (
    pd.DataFrame(RERANK_RESULTS)
    .sort_values(
        [
            "CompleteEvidence@5",
            "EvidenceRecall@5",
            "EvidenceRecall@1"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    RERANK_SUMMARY.style.format({
        "EvidenceRecall@1": "{:.4f}",
        "EvidenceRecall@3": "{:.4f}",
        "EvidenceRecall@5": "{:.4f}",
        "CompleteEvidence@5": "{:.4f}",
    })
)

In [ ]:
# ============================================================
# BASE vs RERANKED
# ============================================================

comparison_rows = []

for key in FINALISTS:

    chunk_config, model, retriever = key

    groups_map = build_gold_evidence_groups(
        dev_questions,
        chunks_by_config[chunk_config]
    )

    for version, rankings in [
        ("Base", DEV_RANKINGS[key]),
        ("Reranked", RERANKED_DEV[key]),
    ]:

        values = []

        for q in dev_questions:

            qid = q["question_id"]
            groups = groups_map[qid]

            values.append({
                "ER1": evidence_recall_at_k(
                    rankings[qid], groups, 1
                ),

                "ER3": evidence_recall_at_k(
                    rankings[qid], groups, 3
                ),

                "ER5": evidence_recall_at_k(
                    rankings[qid], groups, 5
                ),

                "Complete5": complete_evidence_at_k(
                    rankings[qid], groups, 5
                )
            })

        tmp = pd.DataFrame(values)

        comparison_rows.append({
            "chunk": chunk_config,
            "model": model,
            "retriever": retriever,
            "version": version,

            "EvidenceRecall@1":
                tmp["ER1"].mean(),

            "EvidenceRecall@3":
                tmp["ER3"].mean(),

            "EvidenceRecall@5":
                tmp["ER5"].mean(),

            "CompleteEvidence@5":
                tmp["Complete5"].mean(),
        })


BASE_VS_RERANK = pd.DataFrame(
    comparison_rows
)

display(
    BASE_VS_RERANK.style.format({
        "EvidenceRecall@1": "{:.4f}",
        "EvidenceRecall@3": "{:.4f}",
        "EvidenceRecall@5": "{:.4f}",
        "CompleteEvidence@5": "{:.4f}",
    })
)

In [ ]:
# ============================================================
# EXPORT FROZEN DEV CONTEXTS FOR GENERATION BENCHMARK
# ============================================================

import json
from pathlib import Path

FROZEN_RETRIEVAL_KEY = (
    "medium",
    "qwen3_06b",
    "hybrid"
)

# Use the reranked version that achieved:
# EvidenceRecall@5 = 1.0
# CompleteEvidence@5 = 1.0
frozen_rankings = RERANKED_DEV[
    FROZEN_RETRIEVAL_KEY
]

frozen_chunks = chunks_by_config[
    "medium"
]

chunk_lookup = {
    c["chunk_id"]: c
    for c in frozen_chunks
}

generation_records = []

for q in dev_questions:

    qid = q["question_id"]

    top_ids = frozen_rankings[qid][:5]

    contexts = []

    for rank, cid in enumerate(
        top_ids,
        start=1
    ):

        chunk = chunk_lookup[cid]

        contexts.append({
            "source_id": f"S{rank}",
            "rank": rank,
            "chunk_id": cid,
            "document_id": chunk["doc_id"],
            "location_type": chunk["location_type"],
            "location_value": chunk["location_value"],
            "text": chunk["text"],
        })

    generation_records.append({
        "question_id": qid,
        "question": q["question"],
        "language": q["language"],
        "question_type": q["question_type"],
        "answerable": True,
        "expected_answer": q.get(
            "expected_answer"
        ),
        "contexts": contexts,
    })


generation_benchmark = {
    "schema_version": "1.0",

    "split": "DEV",

    "retrieval_frozen": True,

    "retrieval_config": {
        "chunk_config": "medium",
        "chunk_tokens": 320,
        "embedding_model":
            "Qwen/Qwen3-Embedding-0.6B",
        "retrieval":
            "Dense + BM25 RRF",
        "reranker":
            "BAAI/bge-reranker-v2-m3",
        "top_k": 5,
    },

    "records": generation_records,
}


GENERATION_DEV_PATH = (
    WORK_DIR /
    "docmind_generation_dev_frozen.json"
)

with open(
    GENERATION_DEV_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        generation_benchmark,
        f,
        ensure_ascii=False,
        indent=2
    )


print("✅ FROZEN GENERATION BENCHMARK")
print("Questions:", len(generation_records))
print("Contexts/question: 5")
print("Saved:", GENERATION_DEV_PATH)

assert len(generation_records) == 46


In [ ]:
import shutil

# Compress /kaggle/working/docmind_research into docmind_research.zip
shutil.make_archive('docmind_research', 'zip', '/kaggle/working/docmind_research')

In [ ]:
# ============================================================
# FINAL RESEARCH V1 — LOCKED CONFIGURATION
# DO NOT MODIFY AFTER RUNNING THIS
# ============================================================

from pathlib import Path
import json
import gc
import numpy as np
import pandas as pd
import faiss
from rank_bm25 import BM25Okapi

FINAL_CHUNK_CONFIG = "medium"
FINAL_EMBEDDING_KEY = "qwen3_06b"
FINAL_RERANK_TOP_N = 30
FINAL_TOP_K = 5
FINAL_RRF_K = 60

FINAL_CHUNKS = chunks_by_config[FINAL_CHUNK_CONFIG]

NOANSWER_QUESTIONS = [
    q for q in gold_questions
    if not q.get("answerable", False)
]

assert len(dev_questions) == 46, len(dev_questions)
assert len(test_questions) == 12, len(test_questions)
assert len(NOANSWER_QUESTIONS) == 4, len(NOANSWER_QUESTIONS)

# Exact helpers already used during DEV.
for fn in [
    "load_embedding_model",
    "encode_queries",
    "rerank_results_batched",
    "build_gold_evidence_groups",
    "evidence_recall_at_k",
    "complete_evidence_at_k",
]:
    assert fn in globals(), f"Missing existing DEV helper: {fn}"

print("=" * 80)
print("DOCMIND RESEARCH V1 — FINAL FROZEN CONFIG")
print("=" * 80)

print("Embedding      :", "Qwen/Qwen3-Embedding-0.6B")
print("Chunks         :", "320 tokens / 48 overlap")
print("Chunk count    :", len(FINAL_CHUNKS))
print("Retrieval      :", "Dense + BM25")
print("Fusion         :", f"RRF k={FINAL_RRF_K}")
print("Reranker       :", "BAAI/bge-reranker-v2-m3")
print("Rerank top N   :", FINAL_RERANK_TOP_N)
print("Final context K:", FINAL_TOP_K)

print("\nDEV      :", len(dev_questions))
print("TEST     :", len(test_questions))
print("NO-ANSWER:", len(NOANSWER_QUESTIONS))

print("\n🔒 CONFIGURATION FROZEN")

In [ ]:
# ============================================================
# FINAL TEST RETRIEVAL
# ============================================================

# ------------------------------------------------------------
# 1. Find the already-computed Qwen-medium document embeddings
# ------------------------------------------------------------

def find_final_document_embedding():
    candidates = []

    for path in WORK_DIR.rglob("*.npy"):
        try:
            arr = np.load(
                path,
                mmap_mode="r"
            )

            # Document matrix must have exactly one row/chunk.
            if (
                arr.ndim == 2
                and arr.shape[0] == len(FINAL_CHUNKS)
                and arr.shape[1] == 1024
            ):
                candidates.append(path)

        except Exception:
            pass

    # Prefer filenames clearly identifying Qwen + medium.
    preferred = [
        p for p in candidates
        if "qwen" in p.name.lower()
        and "medium" in p.name.lower()
    ]

    if len(preferred) == 1:
        return preferred[0]

    if len(preferred) > 1:
        print("Possible Qwen-medium embeddings:")
        for p in preferred:
            print(" -", p)

        return preferred[0]

    if len(candidates) == 1:
        return candidates[0]

    raise RuntimeError(
        "Could not uniquely identify the cached "
        "Qwen medium document embeddings.\n"
        f"Candidates found: {candidates}"
    )


DOC_EMBED_PATH = find_final_document_embedding()

DOCUMENT_EMBEDDINGS = np.asarray(
    np.load(DOC_EMBED_PATH),
    dtype=np.float32
)

print("Using cached embeddings:")
print(DOC_EMBED_PATH)
print("Shape:", DOCUMENT_EMBEDDINGS.shape)


# ------------------------------------------------------------
# 2. Encode ONLY new TEST / no-answer queries
# using the exact existing Qwen helper from DEV
# ------------------------------------------------------------

embedding_model = load_embedding_model(
    FINAL_EMBEDDING_KEY
)

TEST_QUERY_EMBEDDINGS = encode_queries(
    embedding_model,
    FINAL_EMBEDDING_KEY,
    [q["question"] for q in test_questions],
    batch_size=64
)

NOANSWER_QUERY_EMBEDDINGS = encode_queries(
    embedding_model,
    FINAL_EMBEDDING_KEY,
    [q["question"] for q in NOANSWER_QUESTIONS],
    batch_size=64
)

del embedding_model
gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

TEST_QUERY_EMBEDDINGS = np.asarray(
    TEST_QUERY_EMBEDDINGS,
    dtype=np.float32
)

NOANSWER_QUERY_EMBEDDINGS = np.asarray(
    NOANSWER_QUERY_EMBEDDINGS,
    dtype=np.float32
)

print("TEST query embeddings     :", TEST_QUERY_EMBEDDINGS.shape)
print("NO-ANSWER query embeddings:", NOANSWER_QUERY_EMBEDDINGS.shape)


# ------------------------------------------------------------
# 3. Dense FAISS index
# ------------------------------------------------------------

dense_index = faiss.IndexFlatIP(
    DOCUMENT_EMBEDDINGS.shape[1]
)

dense_index.add(
    DOCUMENT_EMBEDDINGS
)


def dense_rank_questions(
    questions,
    query_embeddings,
    top_k=50,
):
    scores, indices = dense_index.search(
        query_embeddings,
        top_k
    )

    output = {}

    for q, row_indices in zip(
        questions,
        indices
    ):
        output[q["question_id"]] = [
            FINAL_CHUNKS[int(i)]["chunk_id"]
            for i in row_indices
            if i >= 0
        ]

    return output


TEST_DENSE = dense_rank_questions(
    test_questions,
    TEST_QUERY_EMBEDDINGS,
    top_k=50,
)

NOANSWER_DENSE = dense_rank_questions(
    NOANSWER_QUESTIONS,
    NOANSWER_QUERY_EMBEDDINGS,
    top_k=50,
)


# ------------------------------------------------------------
# 4. BM25 — same simple lexical baseline used in research
# ------------------------------------------------------------

def bm25_tokenize(text):
    return re.findall(
        r"\w+",
        str(text).lower(),
        flags=re.UNICODE,
    )


bm25_corpus = [
    bm25_tokenize(c["text"])
    for c in FINAL_CHUNKS
]

FINAL_BM25 = BM25Okapi(
    bm25_corpus
)


def bm25_rank_questions(
    questions,
    top_k=50,
):
    output = {}

    for q in questions:
        scores = FINAL_BM25.get_scores(
            bm25_tokenize(q["question"])
        )

        order = np.argsort(scores)[::-1][
            :top_k
        ]

        output[q["question_id"]] = [
            FINAL_CHUNKS[int(i)]["chunk_id"]
            for i in order
        ]

    return output


TEST_BM25 = bm25_rank_questions(
    test_questions
)

NOANSWER_BM25 = bm25_rank_questions(
    NOANSWER_QUESTIONS
)


# ------------------------------------------------------------
# 5. Reciprocal Rank Fusion
# ------------------------------------------------------------

def rrf_rank(
    dense_ranking,
    bm25_ranking,
    rrf_k=60,
    top_k=50,
):
    scores = {}

    for ranking in [
        dense_ranking,
        bm25_ranking
    ]:
        for rank, chunk_id in enumerate(
            ranking,
            start=1
        ):
            scores[chunk_id] = (
                scores.get(chunk_id, 0.0)
                + 1.0 / (rrf_k + rank)
            )

    return [
        cid
        for cid, _ in sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True,
        )[:top_k]
    ]


def make_hybrid(
    questions,
    dense,
    bm25,
):
    output = {}

    for q in questions:
        qid = q["question_id"]

        output[qid] = rrf_rank(
            dense[qid],
            bm25[qid],
            rrf_k=FINAL_RRF_K,
            top_k=50,
        )

    return output


TEST_HYBRID = make_hybrid(
    test_questions,
    TEST_DENSE,
    TEST_BM25,
)

NOANSWER_HYBRID = make_hybrid(
    NOANSWER_QUESTIONS,
    NOANSWER_DENSE,
    NOANSWER_BM25,
)


# ------------------------------------------------------------
# 6. Frozen BGE reranking
# EXACT helper already used on DEV
# ------------------------------------------------------------

TEST_FINAL_RANKINGS = rerank_results_batched(
    questions=test_questions,
    rankings=TEST_HYBRID,
    chunks=FINAL_CHUNKS,
    top_n=FINAL_RERANK_TOP_N,
    final_k=50,
    batch_size=16,
)

NOANSWER_FINAL_RANKINGS = rerank_results_batched(
    questions=NOANSWER_QUESTIONS,
    rankings=NOANSWER_HYBRID,
    chunks=FINAL_CHUNKS,
    top_n=FINAL_RERANK_TOP_N,
    final_k=50,
    batch_size=16,
)

print("\n✅ FINAL TEST retrieval complete")
print("✅ NO-ANSWER retrieval complete")

In [ ]:
# ============================================================
# FINAL HELD-OUT TEST RETRIEVAL METRICS
# 🔒 DO NOT RETUNE AFTER SEEING THESE RESULTS
# ============================================================

TEST_EVIDENCE_GROUPS = (
    build_gold_evidence_groups(
        test_questions,
        FINAL_CHUNKS,
    )
)

test_metric_rows = []

for q in test_questions:

    qid = q["question_id"]

    ranking = TEST_FINAL_RANKINGS[
        qid
    ]

    groups = TEST_EVIDENCE_GROUPS[
        qid
    ]

    test_metric_rows.append({
        "question_id":
            qid,

        "language":
            q["language"],

        "question_type":
            q["question_type"],

        "EvidenceRecall@1":
            evidence_recall_at_k(
                ranking,
                groups,
                1
            ),

        "EvidenceRecall@3":
            evidence_recall_at_k(
                ranking,
                groups,
                3
            ),

        "EvidenceRecall@5":
            evidence_recall_at_k(
                ranking,
                groups,
                5
            ),

        "CompleteEvidence@5":
            complete_evidence_at_k(
                ranking,
                groups,
                5
            ),
    })


TEST_RETRIEVAL_DF = pd.DataFrame(
    test_metric_rows
)


TEST_RETRIEVAL_SUMMARY = {
    "split":
        "TEST",

    "questions":
        len(TEST_RETRIEVAL_DF),

    "configuration": {
        "embedding":
            "Qwen/Qwen3-Embedding-0.6B",

        "chunk_tokens":
            320,

        "chunk_overlap":
            48,

        "retrieval":
            "dense+BM25",

        "fusion":
            "RRF",

        "rrf_k":
            60,

        "reranker":
            "BAAI/bge-reranker-v2-m3",

        "rerank_top_n":
            30,

        "final_k":
            5,
    },

    "EvidenceRecall@1":
        float(
            TEST_RETRIEVAL_DF[
                "EvidenceRecall@1"
            ].mean()
        ),

    "EvidenceRecall@3":
        float(
            TEST_RETRIEVAL_DF[
                "EvidenceRecall@3"
            ].mean()
        ),

    "EvidenceRecall@5":
        float(
            TEST_RETRIEVAL_DF[
                "EvidenceRecall@5"
            ].mean()
        ),

    "CompleteEvidence@5":
        float(
            TEST_RETRIEVAL_DF[
                "CompleteEvidence@5"
            ].mean()
        ),
}


print("\n")
print("=" * 100)
print("🔒 DOCMIND V1 — FINAL HELD-OUT RETRIEVAL TEST")
print("=" * 100)

for key in [
    "EvidenceRecall@1",
    "EvidenceRecall@3",
    "EvidenceRecall@5",
    "CompleteEvidence@5",
]:
    print(
        f"{key:<25}",
        f"{TEST_RETRIEVAL_SUMMARY[key]:.4f}"
    )


print("\nPer-question results:")
display(TEST_RETRIEVAL_DF)


TEST_RETRIEVAL_DF.to_csv(
    WORK_DIR /
    "test_retrieval_results.csv",
    index=False,
)

with open(
    WORK_DIR /
    "test_retrieval_summary.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TEST_RETRIEVAL_SUMMARY,
        f,
        ensure_ascii=False,
        indent=2,
    )


print("\n✅ Saved:")
print(
    WORK_DIR /
    "test_retrieval_results.csv"
)

print(
    WORK_DIR /
    "test_retrieval_summary.json"
)

print(
    "\n🔒 TEST HAS NOW BEEN OPENED."
)
print(
    "Do NOT change retrieval configuration "
    "based on these numbers."
)

In [ ]:
# ============================================================
# EXPORT FROZEN GENERATION CONTEXTS
# ============================================================

CHUNK_LOOKUP = {
    c["chunk_id"]: c
    for c in FINAL_CHUNKS
}


def make_generation_records(
    questions,
    rankings,
):
    records = []

    for q in questions:

        qid = q["question_id"]

        top_chunks = rankings[qid][
            :FINAL_TOP_K
        ]

        contexts = []

        for rank, chunk_id in enumerate(
            top_chunks,
            start=1,
        ):

            c = CHUNK_LOOKUP[
                chunk_id
            ]

            contexts.append({
                "source_id":
                    f"S{rank}",

                "rank":
                    rank,

                "chunk_id":
                    chunk_id,

                "document_id":
                    c["doc_id"],

                "location_type":
                    c["location_type"],

                "location_value":
                    c["location_value"],

                "text":
                    c["text"],
            })

        records.append({
            "question_id":
                qid,

            "question":
                q["question"],

            "language":
                q["language"],

            "question_type":
                q["question_type"],

            "answerable":
                q["answerable"],

            "expected_answer":
                q.get("expected_answer"),

            "contexts":
                contexts,
        })

    return records


TEST_GENERATION_RECORDS = (
    make_generation_records(
        test_questions,
        TEST_FINAL_RANKINGS,
    )
)

NOANSWER_GENERATION_RECORDS = (
    make_generation_records(
        NOANSWER_QUESTIONS,
        NOANSWER_FINAL_RANKINGS,
    )
)


FROZEN_CONFIG = {
    "embedding":
        "Qwen/Qwen3-Embedding-0.6B",

    "chunk_tokens":
        320,

    "chunk_overlap":
        48,

    "retrieval":
        "dense + BM25",

    "fusion":
        "RRF",

    "rrf_k":
        60,

    "reranker":
        "BAAI/bge-reranker-v2-m3",

    "rerank_top_n":
        30,

    "top_k":
        5,
}


TEST_CONTEXT_PATH = (
    WORK_DIR /
    "docmind_generation_test_frozen.json"
)

NOANSWER_CONTEXT_PATH = (
    WORK_DIR /
    "docmind_generation_noanswer_frozen.json"
)


with open(
    TEST_CONTEXT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        {
            "split": "TEST",
            "frozen": True,
            "retrieval_config":
                FROZEN_CONFIG,
            "records":
                TEST_GENERATION_RECORDS,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )


with open(
    NOANSWER_CONTEXT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        {
            "split": "NOANSWER",
            "frozen": True,
            "retrieval_config":
                FROZEN_CONFIG,
            "records":
                NOANSWER_GENERATION_RECORDS,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )


print("=" * 100)
print("FINAL GENERATION INPUTS EXPORTED")
print("=" * 100)

print("TEST:")
print(TEST_CONTEXT_PATH)

print("\nNO-ANSWER:")
print(NOANSWER_CONTEXT_PATH)

print(
    "\nTEST records:",
    len(TEST_GENERATION_RECORDS)
)

print(
    "NO-ANSWER records:",
    len(NOANSWER_GENERATION_RECORDS)
)

In [73]:
# ============================================================
# DOCMIND V1 — ONE-CELL NO-ANSWER RECOVERY
#
# PURPOSE:
# Reconstruct ONLY the 4 no-answer questions using the frozen
# DocMind pipeline and save:
#
# /kaggle/working/docmind_final_v1/
#     docmind_generation_noanswer_frozen.json
#
# Frozen pipeline:
# Qwen3-Embedding-0.6B
# medium chunks (320 / 48)
# Dense + BM25
# RRF k=60
# BGE-reranker-v2-m3
# rerank top 30
# final top 5
# ============================================================

# ------------------------------------------------------------
# 0. IMPORTS / INSTALL MISSING PACKAGES
# ------------------------------------------------------------

import os
import re
import gc
import sys
import json
import subprocess
import importlib.util

from pathlib import Path

import numpy as np


def ensure_package(package, import_name=None):
    import_name = import_name or package

    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package} ...")

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            package,
        ])


ensure_package(
    "sentence-transformers",
    "sentence_transformers"
)

ensure_package(
    "rank-bm25",
    "rank_bm25"
)

ensure_package(
    "faiss-cpu",
    "faiss"
)


import torch
import faiss

from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi

from sentence_transformers import (
    SentenceTransformer
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)


# ============================================================
# 1. CONSTANTS
# ============================================================

KAGGLE_INPUT = Path(
    "/kaggle/input"
)

OUTPUT_DIR = Path(
    "/kaggle/working/docmind_final_v1"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


EMBEDDING_MODEL_NAME = (
    "Qwen/Qwen3-Embedding-0.6B"
)

RERANKER_NAME = (
    "BAAI/bge-reranker-v2-m3"
)

RRF_K = 60
DENSE_TOP_K = 50
BM25_TOP_K = 50
RERANK_TOP_N = 30
FINAL_K = 5

EXPECTED_CHUNKS = 1210
EXPECTED_DIM = 1024


print("=" * 90)
print("DOCMIND V1 — NO-ANSWER RECOVERY")
print("=" * 90)

print("Output:", OUTPUT_DIR)


# ============================================================
# 2. FIND INPUT FILES AUTOMATICALLY
# ============================================================

def find_exact_file(
    filenames,
    required=True
):
    if isinstance(
        filenames,
        str
    ):
        filenames = [
            filenames
        ]

    matches = []

    for filename in filenames:

        matches.extend(
            KAGGLE_INPUT.rglob(
                filename
            )
        )

    matches = list(
        dict.fromkeys(
            matches
        )
    )

    if not matches:

        if required:
            raise FileNotFoundError(
                "\nCould not find any of:\n"
                +
                "\n".join(
                    filenames
                )
                +
                "\n\nAttach the Kaggle dataset "
                "containing the file."
            )

        return None

    print(
        f"✅ Found: {matches[0]}"
    )

    return matches[0]


# ------------------------------------------------------------
# Gold questions
# ------------------------------------------------------------

GOLD_PATH = find_exact_file([
    "rag_gold_questions.json",
    "rag_gold_questions_starter_v1.json",
])


# ------------------------------------------------------------
# Medium chunks
# ------------------------------------------------------------

MEDIUM_CHUNKS_PATH = find_exact_file([
    "docmind_chunks_medium.json",
    "chunks_medium.json",
    "medium_chunks.json",
])


# ============================================================
# 3. LOAD GOLD QUESTIONS
# ============================================================

with open(
    GOLD_PATH,
    "r",
    encoding="utf-8"
) as f:

    gold_raw = json.load(f)


if isinstance(
    gold_raw,
    list
):

    gold_questions = gold_raw

elif isinstance(
    gold_raw,
    dict
):

    gold_questions = (
        gold_raw.get("questions")
        or
        gold_raw.get("records")
    )

else:

    raise TypeError(
        "Unknown gold JSON structure."
    )


assert isinstance(
    gold_questions,
    list
)


NOANSWER_QUESTIONS = [
    q
    for q in gold_questions
    if not q.get(
        "answerable",
        True
    )
]


print(
    "\nGold questions:",
    len(gold_questions)
)

print(
    "No-answer questions:",
    len(NOANSWER_QUESTIONS)
)


assert len(
    NOANSWER_QUESTIONS
) == 4, (
    f"Expected exactly 4 no-answer questions, "
    f"found {len(NOANSWER_QUESTIONS)}"
)


for q in NOANSWER_QUESTIONS:

    print(
        "\n",
        q["question_id"],
        "|",
        q.get(
            "language"
        ),
        "|",
        q["question"]
    )


# ============================================================
# 4. LOAD MEDIUM CHUNKS
# ============================================================

with open(
    MEDIUM_CHUNKS_PATH,
    "r",
    encoding="utf-8"
) as f:

    chunks_raw = json.load(f)


if isinstance(
    chunks_raw,
    list
):

    FINAL_CHUNKS = chunks_raw

elif isinstance(
    chunks_raw,
    dict
):

    FINAL_CHUNKS = (
        chunks_raw.get("chunks")
        or
        chunks_raw.get("records")
    )

else:

    raise TypeError(
        "Unknown chunks JSON structure."
    )


assert isinstance(
    FINAL_CHUNKS,
    list
)


print(
    "\nMedium chunks:",
    len(FINAL_CHUNKS)
)


assert len(
    FINAL_CHUNKS
) == EXPECTED_CHUNKS, (
    f"Expected {EXPECTED_CHUNKS} medium chunks "
    f"but got {len(FINAL_CHUNKS)}"
)


# ============================================================
# 5. FIND SAVED QWEN MEDIUM DOCUMENT EMBEDDINGS
# ============================================================

embedding_candidates = []


print(
    "\nSearching attached datasets "
    "for the saved 1210 × 1024 matrix..."
)


for path in KAGGLE_INPUT.rglob(
    "*.npy"
):

    try:

        arr = np.load(
            path,
            mmap_mode="r"
        )

        if (
            arr.ndim == 2
            and
            arr.shape == (
                len(FINAL_CHUNKS),
                EXPECTED_DIM
            )
        ):

            embedding_candidates.append(
                path
            )

    except Exception:

        pass


if not embedding_candidates:

    raise FileNotFoundError(
        "\nCould not find the saved "
        "1210 × 1024 document embedding matrix.\n\n"
        "Attach the Kaggle dataset containing "
        "your Qwen medium document embeddings."
    )


qwen_candidates = [

    p
    for p in embedding_candidates

    if "qwen"
    in str(p).lower()
]


if qwen_candidates:

    embedding_candidates = (
        qwen_candidates
    )


print(
    "\nEmbedding candidates:"
)

for p in embedding_candidates:

    print(
        " -",
        p
    )


DOCUMENT_EMBED_PATH = (
    embedding_candidates[0]
)


DOCUMENT_EMBEDDINGS = np.asarray(
    np.load(
        DOCUMENT_EMBED_PATH
    ),
    dtype=np.float32
)


print(
    "\n✅ Using document embeddings:"
)

print(
    DOCUMENT_EMBED_PATH
)

print(
    "Shape:",
    DOCUMENT_EMBEDDINGS.shape
)


assert (
    DOCUMENT_EMBEDDINGS.shape
    ==
    (
        len(FINAL_CHUNKS),
        EXPECTED_DIM
    )
)


# ============================================================
# 6. LOAD QWEN EMBEDDING MODEL
# ============================================================

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(
    "\nLoading embedding model:"
)

print(
    EMBEDDING_MODEL_NAME
)

print(
    "Device:",
    device
)


embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=device,
)


# ============================================================
# 7. EMBED ONLY THE FOUR NO-ANSWER QUERIES
# ============================================================

query_texts = [

    q["question"]

    for q
    in NOANSWER_QUESTIONS
]


# Qwen embedding query instruction used for retrieval queries.
#
# We keep one fixed instruction for all four questions.
task_instruction = (
    "Given a user question, retrieve relevant passages "
    "that answer the question"
)


query_inputs = [

    (
        f"Instruct: {task_instruction}\n"
        f"Query: {question}"
    )

    for question
    in query_texts
]


NOANSWER_QUERY_EMBEDDINGS = (
    embedding_model.encode(
        query_inputs,
        batch_size=16,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )
)


NOANSWER_QUERY_EMBEDDINGS = np.asarray(
    NOANSWER_QUERY_EMBEDDINGS,
    dtype=np.float32
)


print(
    "\nQuery embedding shape:",
    NOANSWER_QUERY_EMBEDDINGS.shape
)


assert (
    NOANSWER_QUERY_EMBEDDINGS.shape
    ==
    (
        4,
        EXPECTED_DIM
    )
)


# Remove embedding model before loading reranker
del embedding_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# 8. DENSE FAISS RETRIEVAL
# ============================================================

print(
    "\nBuilding FAISS index..."
)


dense_index = faiss.IndexFlatIP(
    EXPECTED_DIM
)


dense_index.add(
    DOCUMENT_EMBEDDINGS
)


dense_scores, dense_indices = (
    dense_index.search(
        NOANSWER_QUERY_EMBEDDINGS,
        DENSE_TOP_K
    )
)


NOANSWER_DENSE = {}


for q, indices in zip(
    NOANSWER_QUESTIONS,
    dense_indices
):

    NOANSWER_DENSE[
        q["question_id"]
    ] = [

        FINAL_CHUNKS[
            int(i)
        ]["chunk_id"]

        for i in indices

        if i >= 0
    ]


print(
    "✅ Dense retrieval complete"
)


# ============================================================
# 9. BM25
# ============================================================

def bm25_tokenize(text):

    return re.findall(
        r"\w+",
        str(text).lower(),
        flags=re.UNICODE
    )


print(
    "\nBuilding BM25..."
)


bm25_corpus = [

    bm25_tokenize(
        chunk.get(
            "text",
            ""
        )
    )

    for chunk
    in FINAL_CHUNKS
]


bm25 = BM25Okapi(
    bm25_corpus
)


NOANSWER_BM25 = {}


for q in NOANSWER_QUESTIONS:

    scores = bm25.get_scores(
        bm25_tokenize(
            q["question"]
        )
    )


    order = np.argsort(
        scores
    )[::-1][
        :BM25_TOP_K
    ]


    NOANSWER_BM25[
        q["question_id"]
    ] = [

        FINAL_CHUNKS[
            int(i)
        ]["chunk_id"]

        for i
        in order
    ]


print(
    "✅ BM25 complete"
)


# ============================================================
# 10. RRF FUSION
# ============================================================

def rrf_fuse(
    dense_ranking,
    bm25_ranking,
    rrf_k=60,
    top_k=50
):

    scores = {}


    for ranking in [
        dense_ranking,
        bm25_ranking
    ]:

        for rank, chunk_id in enumerate(
            ranking,
            start=1
        ):

            scores[
                chunk_id
            ] = (

                scores.get(
                    chunk_id,
                    0.0
                )

                +

                1.0 / (
                    rrf_k
                    +
                    rank
                )
            )


    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )


    return [

        chunk_id

        for chunk_id, _
        in ranked[
            :top_k
        ]
    ]


NOANSWER_HYBRID = {}


for q in NOANSWER_QUESTIONS:

    qid = q[
        "question_id"
    ]


    NOANSWER_HYBRID[
        qid
    ] = rrf_fuse(
        NOANSWER_DENSE[qid],
        NOANSWER_BM25[qid],
        rrf_k=RRF_K,
        top_k=50,
    )


print(
    "✅ RRF hybrid retrieval complete"
)


# ============================================================
# 11. LOAD BGE RERANKER
# ============================================================

RERANKER_DEVICE = (
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)


print(
    "\nLoading reranker:"
)

print(
    RERANKER_NAME
)

print(
    "Device:",
    RERANKER_DEVICE
)


reranker_tokenizer = (
    AutoTokenizer.from_pretrained(
        RERANKER_NAME
    )
)


reranker_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        RERANKER_NAME,
        torch_dtype=(
            torch.float16
            if torch.cuda.is_available()
            else torch.float32
        ),
    )
)


reranker_model = reranker_model.to(
    RERANKER_DEVICE
)

reranker_model.eval()


print(
    "✅ Reranker loaded"
)


# ============================================================
# 12. RERANK ONLY 4 × 30 = 120 PAIRS
# ============================================================

chunk_lookup = {

    chunk["chunk_id"]:
        chunk

    for chunk
    in FINAL_CHUNKS
}


pair_records = []


for q in NOANSWER_QUESTIONS:

    qid = q[
        "question_id"
    ]

    query = q[
        "question"
    ]


    for chunk_id in NOANSWER_HYBRID[
        qid
    ][:RERANK_TOP_N]:

        pair_records.append({
            "question_id":
                qid,

            "chunk_id":
                chunk_id,

            "query":
                query,

            "passage":
                chunk_lookup[
                    chunk_id
                ]["text"],
        })


print(
    "\nPairs to rerank:",
    len(pair_records)
)


assert len(
    pair_records
) == (
    len(NOANSWER_QUESTIONS)
    *
    RERANK_TOP_N
)


rerank_scores = []


BATCH_SIZE = 16


for start in tqdm(
    range(
        0,
        len(pair_records),
        BATCH_SIZE
    ),
    desc="Reranking"
):

    batch = pair_records[
        start:
        start + BATCH_SIZE
    ]


    pairs = [

        [
            item["query"],
            item["passage"]
        ]

        for item
        in batch
    ]


    inputs = reranker_tokenizer(
        pairs,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )


    inputs = {

        key:
            value.to(
                RERANKER_DEVICE
            )

        for key, value
        in inputs.items()
    }


    with torch.inference_mode():

        logits = (
            reranker_model(
                **inputs
            )
            .logits
            .view(-1)
            .float()
            .cpu()
            .numpy()
        )


    rerank_scores.extend(
        logits.tolist()
    )


assert len(
    rerank_scores
) == len(
    pair_records
)


# ============================================================
# 13. BUILD FINAL RERANKED RANKINGS
# ============================================================

scores_by_q = {}


for item, score in zip(
    pair_records,
    rerank_scores
):

    qid = item[
        "question_id"
    ]


    scores_by_q.setdefault(
        qid,
        []
    )


    scores_by_q[
        qid
    ].append(
        (
            item["chunk_id"],
            float(score)
        )
    )


NOANSWER_FINAL_RANKINGS = {}


for q in NOANSWER_QUESTIONS:

    qid = q[
        "question_id"
    ]


    reranked_top = [

        chunk_id

        for chunk_id, _
        in sorted(
            scores_by_q[qid],
            key=lambda x: x[1],
            reverse=True
        )
    ]


    # Keep tail unchanged, exactly like DEV logic
    tail = NOANSWER_HYBRID[
        qid
    ][
        RERANK_TOP_N:
    ]


    combined = (
        reranked_top
        +
        tail
    )


    # Deduplicate while preserving order
    seen = set()
    final = []


    for chunk_id in combined:

        if chunk_id not in seen:

            seen.add(
                chunk_id
            )

            final.append(
                chunk_id
            )


    NOANSWER_FINAL_RANKINGS[
        qid
    ] = final


print(
    "✅ Reranking complete"
)


# ============================================================
# 14. BUILD TOP-5 FROZEN GENERATION RECORDS
# ============================================================

NOANSWER_GENERATION_RECORDS = []


for q in NOANSWER_QUESTIONS:

    qid = q[
        "question_id"
    ]


    top_ids = (
        NOANSWER_FINAL_RANKINGS[
            qid
        ][:FINAL_K]
    )


    contexts = []


    for rank, chunk_id in enumerate(
        top_ids,
        start=1
    ):

        chunk = chunk_lookup[
            chunk_id
        ]


        contexts.append({
            "source_id":
                f"S{rank}",

            "rank":
                rank,

            "chunk_id":
                chunk_id,

            "document_id":
                (
                    chunk.get(
                        "doc_id"
                    )
                    or
                    chunk.get(
                        "document_id"
                    )
                ),

            "location_type":
                chunk.get(
                    "location_type"
                ),

            "location_value":
                chunk.get(
                    "location_value"
                ),

            "text":
                chunk.get(
                    "text",
                    ""
                ),
        })


    NOANSWER_GENERATION_RECORDS.append({
        "question_id":
            qid,

        "question":
            q[
                "question"
            ],

        "language":
            q.get(
                "language"
            ),

        "question_type":
            q.get(
                "question_type"
            ),

        "answerable":
            False,

        "expected_answer":
            q.get(
                "expected_answer"
            ),

        "contexts":
            contexts,
    })


assert len(
    NOANSWER_GENERATION_RECORDS
) == 4


for record in (
    NOANSWER_GENERATION_RECORDS
):

    assert len(
        record["contexts"]
    ) == FINAL_K


# ============================================================
# 15. SAVE FINAL FILE
# ============================================================

FROZEN_CONFIG = {

    "embedding":
        EMBEDDING_MODEL_NAME,

    "chunk_size_tokens":
        320,

    "chunk_overlap_tokens":
        48,

    "retrieval":
        "dense + BM25",

    "fusion":
        "RRF",

    "rrf_k":
        RRF_K,

    "reranker":
        RERANKER_NAME,

    "rerank_top_n":
        RERANK_TOP_N,

    "final_k":
        FINAL_K,
}


OUTPUT_PATH = (
    OUTPUT_DIR /
    "docmind_generation_noanswer_frozen.json"
)


payload = {

    "split":
        "NOANSWER",

    "frozen":
        True,

    "retrieval_config":
        FROZEN_CONFIG,

    "records":
        NOANSWER_GENERATION_RECORDS,
}


with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        payload,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 16. VERIFY SAVED FILE
# ============================================================

print("\n")
print("=" * 90)
print("✅ NO-ANSWER RECOVERY COMPLETE")
print("=" * 90)

print(
    "Saved:"
)

print(
    OUTPUT_PATH
)

print(
    "\nQuestions:",
    len(NOANSWER_GENERATION_RECORDS)
)

print(
    "Contexts per question:",
    FINAL_K
)

print(
    "File size:",
    round(
        OUTPUT_PATH.stat().st_size
        / 1024,
        2
    ),
    "KB"
)


print(
    "\nQuestions recovered:"
)

for record in (
    NOANSWER_GENERATION_RECORDS
):

    print(
        " -",
        record[
            "question_id"
        ],
        "|",
        record[
            "language"
        ],
        "|",
        record[
            "question"
        ]
    )


print(
    "\n🎉 DONE."
)

print(
    "Now save/download this ONE file and "
    "attach it to Notebook 03."
)

DOCMIND V1 — NO-ANSWER RECOVERY
Output: /kaggle/working/docmind_final_v1
✅ Found: /kaggle/input/datasets/ripperdzz/past-session/rag_gold_questions.json
✅ Found: /kaggle/input/datasets/ripperdzz/past-session/docmind_chunks_medium.json

Gold questions: 62
No-answer questions: 4

 q059 | en | What was the exact electricity cost of training the transformer model described in these documents?

 q060 | fr | Quel est le numéro de téléphone personnel de l'enseignant du cours de régression ?

 q061 | en | What GPU model was used to train the examples in the RNN lecture?

 q062 | fr | Quel est le salaire annuel de l'auteur du cours SVM ?

Medium chunks: 1210

Searching attached datasets for the saved 1210 × 1024 matrix...

Embedding candidates:
 - /kaggle/input/datasets/ripperdzz/past-session/qwen3_06b__medium__documents.npy

✅ Using document embeddings:
/kaggle/input/datasets/ripperdzz/past-session/qwen3_06b__medium__documents.npy
Shape: (1210, 1024)

Loading embedding model:
Qwen/Qwen3-Embeddi

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...


Query embedding shape: (4, 1024)

Building FAISS index...
✅ Dense retrieval complete

Building BM25...
✅ BM25 complete
✅ RRF hybrid retrieval complete

Loading reranker:
BAAI/bge-reranker-v2-m3
Device: cuda:0


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ Reranker loaded

Pairs to rerank: 120


Reranking:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Reranking complete


✅ NO-ANSWER RECOVERY COMPLETE
Saved:
/kaggle/working/docmind_final_v1/docmind_generation_noanswer_frozen.json

Questions: 4
Contexts per question: 5
File size: 13.16 KB

Questions recovered:
 - q059 | en | What was the exact electricity cost of training the transformer model described in these documents?
 - q060 | fr | Quel est le numéro de téléphone personnel de l'enseignant du cours de régression ?
 - q061 | en | What GPU model was used to train the examples in the RNN lecture?
 - q062 | fr | Quel est le salaire annuel de l'auteur du cours SVM ?

🎉 DONE.
Now save/download this ONE file and attach it to Notebook 03.
